In [4]:
import os
import time
import folium
import tomllib
import shapely
import sqlparse
import warnings
import pymysql
import numpy as np
import pandas as pd
import datetime as dt
from tqdm import tqdm
from dotenv import load_dotenv
from shapely.geometry import Polygon
from shapely import points, contains, prepare, is_prepared

# Vectorization vs Parallelization

For June 2026, the seismic revision routine makes a search of 17 different checks on the seismic data. Each check is a function that takes in the seismic data and performs some operations on it to check for certain conditions. The checks are performed sequentially, which means that each check is performed one after the other. This can be time-consuming, especially if the seismic data is large. In order to handle this, initially parallelization was implemented using the multiprocessing library in Python. This allowed the checks to be performed in parallel, which significantly reduced the time taken to perform the checks. However, this approach had some limitations, such as the overhead of creating and managing multiple processes, and the need to ensure that the checks were thread-safe.

Furthermore, there are some operations that are performed on the seismic data that can be vectorized using libraries such as NumPy. Vectorization allows for the operations to be performed on entire arrays of data at once, rather than iterating through each element individually. This can significantly reduce the time taken to perform the operations, as it takes advantage of the underlying hardware optimizations for array operations. In addition, there is no need to initialize multiple workers or manage the overhead associated with parallelization. By using vectorization, we can reduce energy consumption and improve the efficiency of the seismic revision routine, while also simplifying the code and reducing the potential for errors. Overall, while parallelization can be useful in certain situations, we will check if vectorization is a more efficient and effective approach for handling large datasets and performing complex operations on them.

In this notebook, we will compare the performance of vectorization and parallelization for a specific check in the seismic revision routine. We will implement both approaches and measure the time taken to perform the check on a sample seismic dataset. We will also analyze the results and discuss the advantages and disadvantages of each approach. Finally, we will make recommendations on which approach to use for different scenarios in the seismic revision routine. The main idea is to define single functions to perform each check, and then use either vectorization or parallelization to apply those functions to the seismic data.

## Query seismic data from database

Let's start by querying the seismic data from the database. We will use the `pymysql` library to connect to the database and execute a SQL query to retrieve the seismic data. We will also use the `dotenv` library to load the database credentials from a `.env` file, as stated in the previous notebook.

In [5]:
load_dotenv(dotenv_path=os.path.join(os.getcwd(), '.env'))

# Cutover date: SC3 → SC6
SC6_CUTOVER = dt.datetime(2026, 3, 17, 0, 0, 0)

def _build_connection(prefix: str):
    """Create a pymysql connection using .env credentials for a given prefix."""
    return pymysql.connect(
        host=os.getenv(f'SERVER_{prefix}_HOST'),
        port=int(os.getenv(f'SERVER_{prefix}_PORT', 3306)),
        user=os.getenv(f'SERVER_{prefix}_USERNAME'),
        password=os.getenv(f'SERVER_{prefix}_PASSWORD'),
        database=os.getenv(f'SERVER_{prefix}_DATABASE'),
    )


def _query_db(
    prefix: str,
    query: str,
    start_time: dt.datetime,
    end_time: dt.datetime,
    desc: str,
    **kwargs,
) -> pd.DataFrame:
    """
    Execute a time-bounded SQL query against a single database.

    Parameters
    ----------
    prefix : str
        Credential prefix — 'SC3' or 'SC6'.
    query : str
        Base SQL query ending before the BETWEEN clause.
    start_time, end_time : datetime
        Time bounds for the query.
    desc : str
        Label shown in the tqdm progress bar.
    """
    start_str = start_time.strftime("%Y-%m-%d %H:%M:%S")
    end_str   = end_time.strftime("%Y-%m-%d %H:%M:%S")
    full_query = (
        f"{query} '{start_str}' AND '{end_str}' "
        f"ORDER BY Origin.time_value ASC;"
    )

    with warnings.catch_warnings():
        warnings.simplefilter("ignore", UserWarning)
        conn = _build_connection(prefix)
        try:
            with tqdm(
                total=1,
                desc=desc,
                unit="query",
                leave=False,
                bar_format="{desc}",
            ) as pbar:
                df = pd.read_sql_query(full_query, conn, **kwargs)
                pbar.update(1)
        finally:
            conn.close()

    return df

def _to_naive_utc(t: dt.datetime) -> dt.datetime:
    """
    Normalize a datetime to naive UTC.
    - Timezone-aware → convert to UTC, strip tzinfo
    - Naive → assumed UTC already, returned as-is
    """
    if t.tzinfo is not None:
        return t.astimezone(dt.timezone.utc).replace(tzinfo=None)
    return t

def connect_to_db(
    query: str,
    start_time: dt.datetime = None,
    end_time: dt.datetime = None,
    **kwargs,
) -> pd.DataFrame:
    """
    Query SC3 and/or SC6 databases depending on the requested time range.

    Decision logic:
        - end_time   <= SC6_CUTOVER  → SC3 only
        - start_time >= SC6_CUTOVER  → SC6 only
        - start_time <  SC6_CUTOVER  < end_time → both (with overlap warning)

    Parameters
    ----------
    query : str
        Base SQL query ending before the BETWEEN clause.
    start_time : datetime, optional
        Start of the time range (UTC). If None, runs the query as-is.
    end_time : datetime, optional
        End of the time range (UTC). If None, runs the query as-is.

    Returns
    -------
    pd.DataFrame
        Query results, merged and sorted by time_value when both DBs are hit.
    """
    # --- No time range: run query as-is against SC3 (legacy default) ----------
    if start_time is None or end_time is None:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", UserWarning)
            conn = _build_connection("SC3")
            try:
                with tqdm(
                    total=1,
                    desc="Querying database...",
                    unit="query",
                    leave=False,
                    bar_format="{desc}",
                ) as pbar:
                    df = pd.read_sql_query(query, conn, **kwargs)
                    pbar.update(1)
            finally:
                conn.close()
        return df

    # --- Normalize to naive UTC before any comparison ------------------------
    start_time = _to_naive_utc(start_time)
    end_time   = _to_naive_utc(end_time)

    # --- Determine which databases are needed --------------------------------
    only_sc3 = end_time   <= SC6_CUTOVER
    only_sc6 = start_time >= SC6_CUTOVER
    both     = not only_sc3 and not only_sc6       # straddles the cutover

    if both:
        warnings.warn(
            f"\n[DATABASE WARNING] The requested time range "
            f"({start_time:%Y-%m-%d %H:%M:%S} → {end_time:%Y-%m-%d %H:%M:%S}) "
            f"spans the seiscomp3-seiscomp6 database cutover ({SC6_CUTOVER:%Y-%m-%d %H:%M:%S} UTC). "
            f"Both databases will be queried and results merged.\n",
            UserWarning,
            stacklevel=2,
        )

    # --- SC3 only ------------------------------------------------------------
    if only_sc3:
        return _query_db(
            prefix="SC3",
            query=query,
            start_time=start_time,
            end_time=end_time,
            desc="Querying SC3 database...",
            **kwargs,
        )

    # --- SC6 only ------------------------------------------------------------
    if only_sc6:
        return _query_db(
            prefix="SC6",
            query=query,
            start_time=start_time,
            end_time=end_time,
            desc="Querying SC6 database...",
            **kwargs,
        )

    # --- Both databases (straddles cutover) ----------------------------------
    # SC3: [start_time, SC6_CUTOVER)
    # SC6: [SC6_CUTOVER, end_time]
    df_sc3 = _query_db(
        prefix="SC3",
        query=query,
        start_time=start_time,
        end_time=SC6_CUTOVER,
        desc="Querying SC3 database (1/2)...",
        **kwargs,
    )
    df_sc6 = _query_db(
        prefix="SC6",
        query=query,
        start_time=SC6_CUTOVER,
        end_time=end_time,
        desc="Querying SC6 database (2/2)...",
        **kwargs,
    )

    # Merge and re-sort by time_value
    df = (
        pd.concat([df_sc3, df_sc6], ignore_index=True)
        .sort_values("time_value")
        .reset_index(drop=True)
    )

    return df

In [6]:
# Read revision.sql file
with open('./queries/revision.sql', 'r') as file:
    revision_query = file.read()

clean_sql = sqlparse.format(revision_query, strip_comments=True).strip()

initial_time = dt.datetime(2026, 8, 10, 0, 0, 0)
final_time = dt.datetime.now(dt.UTC)
event_df3 = connect_to_db(clean_sql, start_time=initial_time, end_time=final_time)

print(f"Number of rows in the seismic data: {len(event_df3)}")
# Print how events are by event_type
print("Number of events by event_type:")
print(event_df3['event_type'].value_counts())

Number of rows in the seismic data: 5741
Number of events by event_type:
event_type
not locatable                  3070
earthquake                     2544
outside of network interest      69
explosion                        51
not existing                      5
Name: count, dtype: int64


In [7]:
# Check size in memory of the seismic data
memory_usage = event_df3.memory_usage(deep=True).sum() / (1024 ** 2)
print(f"Memory usage of the seismic data: {memory_usage:.2f} MB")

Memory usage of the seismic data: 3.36 MB


## 1. Numeric Comparison checks

The seismic revision routine performs a list of quality checks on earthquakes, based on relational or absolute thresholds. These checks are designed to identify earthquakes that may not be reliable and may require further investigation. Some of the checks that are performed include:

1. High RMS: This check identifies earthquakes with a high root-mean-square (RMS) value, which indicates that the seismic data is noisy and may not be reliable. The threshold for this check is typically set at a certain value, such as 1.51.
2. Localization uncertainty: This check identifies earthquakes with a high localization uncertainty, which indicates that the location of the earthquake is not well-defined. The threshold for this check is typically set at a certain value, such as 12 km. It is applied both on latitude, longitude, and depth.
3. Depth check: This check identifies earthquakes with a depth that is outside of a certain range, such as between 0 and 700 km. This check is important because earthquakes that are too shallow or too deep may not be reliable and may require further investigation.

For all these type of checks, it is possible to vectorize the solution by applying the check to the entire dataset at once, rather than iterating through each earthquake individually. The idea here is to create a single general function, receiving the filtered seismic data, the threshold value or values (if there are multiple thresholds), and the column to be checked. The function will then apply the check to the entire dataset and return a boolean mask indicating which earthquakes meet the criteria for being flagged as unreliable. This approach can significantly reduce the time taken to perform the checks, without making it too complicated to be debugged or maintained.

In [8]:
# Previous version
def single_check(event):
    observations = []

    # First check: High RMS values
    exceptions = ["not locatable", "outside of network interest", "volcanic eruption", "explosion", "not existing"]
    if event['quality_standardError'] > 1.51 and event['event_type'] not in exceptions:
        observations.append("High RMS value")

    if len(observations) > 0:  # If the event has observations, return the information
        return event, observations
    else:
        return None, None

# For loop version
time1 = time.time()
results = []
for _, event in event_df3.iterrows():
    result, obs = single_check(event)
    if result is not None:
        results.append((result, obs))
high_rms_df_loop = pd.DataFrame([res[0] for res in results])
time2 = time.time()
print(f"Number of events with high RMS (loop version): {len(high_rms_df_loop)}")
print(f"Time taken for high RMS check (loop version): {time2 - time1:.4f} seconds")

Number of events with high RMS (loop version): 0
Time taken for high RMS check (loop version): 0.4466 seconds


In [9]:
# Vectorized function to make comparison between a column and a threshold value
def build_quality_mask(
    events: pd.DataFrame,
    column: str,
    mode: str,
    threshold=None,
    lower=None,
    upper=None,
    dtype=np.float64
) -> np.ndarray:
    """
    Vectorized generic comparator for seismic quality checks.

    Parameters
    ----------
    events : pd.DataFrame
        Input seismic dataframe.
    column : str
        Column to evaluate.
    mode : str
        Comparison mode:
            'gt'       -> values > threshold
            'ge'       -> values >= threshold
            'lt'       -> values < threshold
            'le'       -> values <= threshold
            'eq'       -> values == threshold
            'ne'       -> values != threshold
            'between'  -> lower <= values <= upper
            'outside'  -> values < lower or values > upper
            'abs_gt'   -> abs(values) > threshold
            'abs_ge'   -> abs(values) >= threshold
    threshold : float, optional
        Threshold for one-sided and equality comparisons.
    lower, upper : float, optional
        Bounds for range comparisons.
    dtype : numpy dtype
        Target dtype for NumPy conversion.

    Returns
    -------
    np.ndarray
        Boolean mask of flagged rows.
    """
    values = events[column].to_numpy(dtype=dtype, copy=False)

    if mode == 'gt':
        return values > threshold
    elif mode == 'ge':
        return values >= threshold
    elif mode == 'lt':
        return values < threshold
    elif mode == 'le':
        return values <= threshold
    elif mode == 'eq':
        return values == threshold
    elif mode == 'ne':
        return values != threshold
    elif mode == 'between':
        return (values >= lower) & (values <= upper)
    elif mode == 'outside':
        return (values < lower) | (values > upper)
    elif mode == 'abs_gt':
        return np.abs(values) > threshold
    elif mode == 'abs_ge':
        return np.abs(values) >= threshold
    else:
        raise ValueError(f"Unsupported mode: {mode!r}")

# Check for high RMS values on 'earthquake' and 'volcanic eruption' event types
time1 = time.time()
rms_threshold = 1.51
rms_mask = build_quality_mask(
    events=event_df3[event_df3['event_type'].isin(['earthquake', 'volcanic eruption'])],
    column='quality_standardError',
    mode='gt',
    threshold=rms_threshold
)
high_rms_df = event_df3[event_df3['event_type'].isin(['earthquake', 'volcanic eruption'])][rms_mask]
time2 = time.time()
print(f"Number of events with high RMS: {len(high_rms_df)}")
print(f"Time taken for high RMS check: {time2 - time1:.4f} seconds")

Number of events with high RMS: 0
Time taken for high RMS check: 0.0147 seconds


As you can see, the vectorized version of the high RMS check is significantly faster than the loop version (13.53 s to just 0.0812 s!). This is because the vectorized version takes advantage of NumPy's optimized array operations, which are implemented in C and can be executed much faster than Python loops. In contrast, the loop version iterates through each event one by one, which is much slower, especially for large datasets. Additionally, the vectorized version is more concise and easier to read, as it eliminates the need for explicit loops and conditional statements. Overall, this demonstrates the significant performance benefits of using vectorization for data processing tasks in Python.

Now let's create a wrapper function to apply multiple checks at once:

In [10]:
# Wrapper function to apply multiple checks at once
def seismic_quality_checks(events: pd.DataFrame) -> pd.DataFrame:
    """
    Apply common earthquake quality checks and return flagged events.
    """
    selections = events[events['event_type'].eq('earthquake')].reset_index(drop=True)

    if selections.empty:
        return selections.iloc[0:0].copy()

    masks = {
        'High RMS': build_quality_mask(
            selections, column='quality_standardError', mode='gt', threshold=1.51
        ),
        'High err_lat': build_quality_mask(
             selections, column='latitude_uncertainty', mode='gt', threshold=12.0
        ),
        'High err_lon': build_quality_mask(
            selections, column='longitude_uncertainty', mode='gt', threshold=12.0
        ),
        'High err_depth': build_quality_mask(
            selections, column='depth_uncertainty', mode='gt', threshold=12.0
        ),
        'Invalid depth': build_quality_mask(
            selections, column='depth_value', mode='outside', lower=0.0, upper=200.0
        ),
        'Earthquake with 6 or less phase count' : build_quality_mask(
            selections, column='quality_associatedPhaseCount', mode='le', threshold=6.0
        )
    }

    combined_mask = np.zeros(len(selections), dtype=bool)
    for mask in masks.values():
        combined_mask |= mask

    flagged = selections.loc[combined_mask].copy()

    flagged_idx = np.where(combined_mask)[0]
    observations = []
    for i in flagged_idx:
        obs = [name for name, mask in masks.items() if mask[i]]
        observations.append(', '.join(obs))

    flagged['Observations'] = observations
    return flagged.reset_index(drop=True)

# Apply the checks and measure time
time1 = time.time()
flagged_events = seismic_quality_checks(event_df3)
time2 = time.time()
print(f"Number of flagged events: {len(flagged_events)}")
print(f"Time taken for seismic quality checks: {time2 - time1:.4f} seconds")

Number of flagged events: 6
Time taken for seismic quality checks: 0.0064 seconds


The wrapper function `seismic_quality_checks` applies multiple quality checks to the seismic data and returns a DataFrame of flagged events along with the observations for each event. The function first filters the input DataFrame to include only earthquake events, and then applies each check using the `build_quality_mask` function. The results of all checks are combined into a single boolean mask, which is used to select the flagged events. Finally, the observations for each flagged event are compiled into a new column in the resulting DataFrame.

Now, it is time to add more checks to the wrapper function in order to make it more comprehensive. To maintain the readability and simplicity of the code, we will use a TOML file to store the configuration for each check, including the column and event types to be checked, the mode of comparison, and the threshold values. This way, we can easily add or modify checks without having to change the code of the wrapper function itself. The wrapper function will read the configuration from the TOML file and apply the checks accordingly. This approach allows us to keep the code clean and maintainable while still providing a flexible way to manage the quality checks for seismic events.

In [11]:
# Read TOML file without comments
with open('./seismic_checks.toml', 'rb') as f:
    checks_config = tomllib.load(f)["checks"]

checks_config

[{'name': 'High RMS',
  'column': 'quality_standardError',
  'mode': 'ge',
  'threshold': 1.5,
  'event_type': ['earthquake', 'explosion', 'volcanic eruption']},
 {'name': 'High Latitude Uncertainty',
  'column': 'latitude_uncertainty',
  'mode': 'gt',
  'threshold': 12,
  'event_type': ['earthquake', 'explosion', 'volcanic eruption']},
 {'name': 'High Longitude Uncertainty',
  'column': 'longitude_uncertainty',
  'mode': 'gt',
  'threshold': 12,
  'event_type': ['earthquake', 'explosion', 'volcanic eruption']},
 {'name': 'High Depth Uncertainty',
  'column': 'depth_uncertainty',
  'mode': 'gt',
  'threshold': 12,
  'event_type': ['earthquake', 'volcanic eruption']},
 {'name': 'Negative Depth',
  'column': 'depth_value',
  'mode': 'lt',
  'threshold': 0,
  'event_type': ['earthquake',
   'explosion',
   'volcanic eruption',
   'not locatable',
   'outside of network interest']},
 {'name': 'Noncommon High Depth',
  'column': 'depth_value',
  'mode': 'ge',
  'threshold': 200,
  'event_ty

In [12]:
def load_checks(path: str = "seismic_checks.toml") -> list[dict]:
    """Load quality checks config from a TOML file."""
    with open(path, "rb") as f:
        return tomllib.load(f)["checks"]


def seismic_quality_checks(
    events: pd.DataFrame,
    checks_path: str = "seismic_checks.toml",
) -> pd.DataFrame:
    """
    Apply seismic quality checks loaded from a TOML config file.

    Each check specifies its own target event_type list, so different checks
    can apply to different subsets of the dataset. The special keyword "all"
    means the check applies to every event type present in the data.

    Parameters
    ----------
    events : pd.DataFrame
        Input seismic dataframe.
    checks_path : str
        Path to the TOML config file.

    Returns
    -------
    pd.DataFrame
        Flagged events with an 'Observations' column listing all triggered
        checks per event. Each publicID appears at most once.
    """
    checks = load_checks(checks_path)

    if events.empty:
        return events.iloc[0:0].copy()

    # observations_map: { row_index -> [check_name, ...] }
    observations_map: dict[int, list[str]] = {}

    for check in checks:
        # Select only rows matching this check's event types
        subset = events[events["event_type"].isin(check["event_type"])]

        if subset.empty:
            continue

        # Build the quality mask on the subset using original index
        kwargs = {k: v for k, v in check.items() if k not in ("name", "event_type")}
        flagged_mask = build_quality_mask(subset, **kwargs)

        # Map triggered rows back to original DataFrame index
        flagged_original_idx = subset.index[flagged_mask]
        for idx in flagged_original_idx:
            observations_map.setdefault(idx, []).append(check["name"])

    if not observations_map:
        return events.iloc[0:0].copy()

    # Build output from all flagged original indices — each event appears once
    flagged_idx = sorted(observations_map.keys())
    flagged = events.loc[flagged_idx].copy()
    flagged["Observations"] = [
        ", ".join(observations_map[i]) for i in flagged_idx
    ]

    return flagged.reset_index(drop=True)

In [13]:
time1 = time.time()
flagged = seismic_quality_checks(event_df3)
time2 = time.time()
print(f"Number of flagged events: {len(flagged)}")
print(f"Time taken for seismic quality checks with TOML config: {time2 - time1:.4f} seconds")

Number of flagged events: 16
Time taken for seismic quality checks with TOML config: 0.0455 seconds


By this, we have successfully implemented a vectorized approach to perform seismic quality checks on the dataset, and we have also made the checks configurable through a TOML file. This allows us to easily add or modify checks without changing the code of the wrapper function, while still maintaining readability and performance. The time taken to perform the checks is significantly reduced compared to a loop-based approach, and the code is more concise and easier to maintain. Overall, this demonstrates the benefits of using vectorization and configuration files for data processing tasks in Python.

## 1.1. Locatable events: The Main Challenge of Single Comparisons

A check of the seismic revision routine is to identify events that are locatable, but are labeled incorrectly as "not locatable". At RSNC, an event is locatable if it has at least 4 p phases and 2 s phases associated to it and are used to the localization model. This check is important because it can help to identify earthquakes that may have been misclassified and may require further investigation.

However, the previous routine only checks a verification on the _'quality_associatedPhaseCount'_ column to be greater than or equal to 8 (plus one due to an event with 6 phases in seiscomp3 will have a _quality_associatedPhaseCount_ of 7). Then, an event with for example 8 p phases and 0 s phases would be flagged as locatable, which is not correct.

The challenge here is that query the number of p and s phases associated to each event requires a join between the _Origin_ and _Arrival_ tables in the database, which can be time-consuming, especially for large datasets. Additionally, the check needs to be performed for each event individually, which can further increase the time taken to perform the check. Therefore, the strategy here is to use the columns _quality_associatedPhaseCount_, _quality_usedPhaseCount_, _quality_usedStationCount_ and _quality_associatedStationCount_ to create a vectorized check that can identify locatable events without the need for a join between the tables. This approach can significantly reduce the time taken to perform the check, while still providing accurate results.

The main idea of thi section is to check how the columns _quality_associatedPhaseCount_, _quality_usedPhaseCount_, _quality_usedStationCount_ and _quality_associatedStationCount_ are populated for different events in both seiscomp3 and seiscomp6 databases, and then use this information to create a vectorized check that can identify locatable events based on the number of p and s phases associated to each event. We will analyze the data and see if there are any patterns or thresholds that can be used to identify locatable events, and then implement the check accordingly.

### Seiscomp3

At Seiscomp3, there is a huge problem with the quality columns, as they are not populated correctly. For example, consider the following examples:

In [14]:
# First example: 3 p and 3 s event (within 2026-03-01 11:13:00 and 2026-03-01 11:14:00)
start_filter = dt.datetime(2026, 3, 1, 11, 13, 0)
end_filter = dt.datetime(2026, 3, 1, 11, 14, 0)
subset_df = event_df3[(event_df3['time_value'] >= start_filter) & (event_df3['time_value'] <= end_filter)].copy()
subset_df[['time_value', 'event_type', 'quality_associatedPhaseCount', 'quality_usedPhaseCount', 'quality_usedStationCount', 'quality_associatedStationCount']]

,time_value,event_type,quality_associatedPhaseCount,quality_usedPhaseCount,quality_usedStationCount,quality_associatedStationCount


In [15]:
# Second example: 4 p and 3 s event (within 2026-03-01 06:38:00 and 2026-03-01 06:39:00)
start_filter_2 = dt.datetime(2026, 3, 1, 6, 38, 0)
end_filter_2 = dt.datetime(2026, 3, 1, 6, 39, 0)
subset_df_2 = event_df3[(event_df3['time_value'] >= start_filter_2) & (event_df3['time_value'] <= end_filter_2)].copy()
subset_df_2[['time_value', 'event_type', 'quality_associatedPhaseCount', 'quality_usedPhaseCount', 'quality_usedStationCount', 'quality_associatedStationCount']]

,time_value,event_type,quality_associatedPhaseCount,quality_usedPhaseCount,quality_usedStationCount,quality_associatedStationCount


In [16]:
# Third example: 4 p and 4 s event (within 2026-03-01 00:22:00 and 2026-03-01 00:23:00)
start_filter_3 = dt.datetime(2026, 3, 1, 0, 22, 0)
end_filter_3 = dt.datetime(2026, 3, 1, 0, 23, 0)
subset_df_3 = event_df3[(event_df3['time_value'] >= start_filter_3) & (event_df3['time_value'] <= end_filter_3)].copy()
subset_df_3[['time_value', 'event_type', 'quality_associatedPhaseCount', 'quality_usedPhaseCount', 'quality_usedStationCount', 'quality_associatedStationCount']]

,time_value,event_type,quality_associatedPhaseCount,quality_usedPhaseCount,quality_usedStationCount,quality_associatedStationCount


In [17]:
# Fourth example: 4 p and 4 s event (within 2026-03-01 11:33:00 and 2026-03-01 11:34:00)
start_filter_4 = dt.datetime(2026, 3, 1, 11, 33, 0)
end_filter_4 = dt.datetime(2026, 3, 1, 11, 34, 0)
subset_df_4 = event_df3[(event_df3['time_value'] >= start_filter_4) & (event_df3['time_value'] <= end_filter_4)].copy()
subset_df_4[['time_value', 'event_type', 'quality_associatedPhaseCount', 'quality_usedPhaseCount', 'quality_usedStationCount', 'quality_associatedStationCount']]

,time_value,event_type,quality_associatedPhaseCount,quality_usedPhaseCount,quality_usedStationCount,quality_associatedStationCount


In [18]:
# Fifth example: 44 p (43 used) and 40 s (81 used picks to locate and 84 total picks) event (within 2026-03-01 09:37:00 and 2026-03-01 09:38:00)
start_filter_5 = dt.datetime(2026, 3, 1, 9, 37, 0)
end_filter_5 = dt.datetime(2026, 3, 1, 9, 38, 0)
subset_df_5 = event_df3[(event_df3['time_value'] >= start_filter_5) & (event_df3['time_value'] <= end_filter_5)].copy()
subset_df_5[['time_value', 'event_type', 'quality_associatedPhaseCount', 'quality_usedPhaseCount', 'quality_usedStationCount', 'quality_associatedStationCount']]

,time_value,event_type,quality_associatedPhaseCount,quality_usedPhaseCount,quality_usedStationCount,quality_associatedStationCount


Summarizing the results for seiscomp3, we have:

| Event | p phases | s phases | quality_associatedPhaseCount | quality_usedPhaseCount | quality_usedStationCount | quality_associatedStationCount |
|------|----------|----------|------------------------------|------------------------|--------------------------|--------------------------------|
| 1    | 3        | 3        | 7                            | 7                      | 6                        | NaN                            |
| 2    | 4        | 3        | 8                            | 8                      | 7                        | NaN                            |
| 3    | 4        | 4        | 8                            | 8                      | 4                        | NaN                            |
| 4    | 4        | 4        | 9                            | 9                      | 8                        | NaN                            |
| 5    | 44       | 40       | 84                           | 81                     | 43                        | NaN                            |

As we can see, the columns _quality_associatedPhaseCount_, _quality_usedPhaseCount_, _quality_usedStationCount_ and _quality_associatedStationCount_ are not populated correctly in seiscomp3, and they do not reflect the actual number of p and s phases associated to each event. This makes it impossible to use these columns to create a vectorized check for locatable events in seiscomp3, as they do not provide accurate information about the number of phases associated to each event. Moreover, as you can see from the table, the 'quality_associatedStationCount' column is not populated at all, which further confirms the issue with the quality columns in seiscomp3!


### Seiscomp6

With the new implementation of seiscomp6 since 2026-03-17 00:00:00 UTC, the main question is: are the columns _quality_associatedPhaseCount_, _quality_usedPhaseCount_, _quality_usedStationCount_ and _quality_associatedStationCount_ populated correctly in seiscomp6, and do they reflect the actual number of p and s phases associated to each event? Let' see again some examples:

In [19]:
# Sixth example: 72 p and 72 s event (within 2026-06-01 04:59:00 and 2026-06-01 05:00:00) 130/144 used phases
start_filter_6 = dt.datetime(2026, 6, 1, 4, 59, 0)
end_filter_6 = dt.datetime(2026, 6, 1, 5, 0, 0)
subset_df_6 = event_df3[(event_df3['time_value'] >= start_filter_6) & (event_df3['time_value'] <= end_filter_6)].copy()
subset_df_6[['time_value', 'event_type', 'quality_associatedPhaseCount', 'quality_usedPhaseCount', 'quality_usedStationCount', 'quality_associatedStationCount']]

,time_value,event_type,quality_associatedPhaseCount,quality_usedPhaseCount,quality_usedStationCount,quality_associatedStationCount


In [20]:
# Seventh example: 4 p and 4 s event (within 2026-06-01 00:25:00 and 2026-06-01 00:26:00)
start_filter_7 = dt.datetime(2026, 6, 1, 0, 25, 0)
end_filter_7 = dt.datetime(2026, 6, 1, 0, 26, 0)
subset_df_7 = event_df3[(event_df3['time_value'] >= start_filter_7) & (event_df3['time_value'] <= end_filter_7)].copy()
subset_df_7[['time_value', 'event_type', 'quality_associatedPhaseCount', 'quality_usedPhaseCount', 'quality_usedStationCount', 'quality_associatedStationCount']]

,time_value,event_type,quality_associatedPhaseCount,quality_usedPhaseCount,quality_usedStationCount,quality_associatedStationCount


In [21]:
# Eighth example: 4 p and 3 s event (within 2026-06-01 07:50:00 and 2026-06-01 07:51:00)
start_filter_8 = dt.datetime(2026, 6, 1, 7, 50, 0)
end_filter_8 = dt.datetime(2026, 6, 1, 7, 50, 4)
subset_df_8 = event_df3[(event_df3['time_value'] >= start_filter_8) & (event_df3['time_value'] <= end_filter_8)].copy()
subset_df_8[['time_value', 'event_type', 'quality_associatedPhaseCount', 'quality_usedPhaseCount', 'quality_usedStationCount', 'quality_associatedStationCount']]

,time_value,event_type,quality_associatedPhaseCount,quality_usedPhaseCount,quality_usedStationCount,quality_associatedStationCount


In [22]:
# Ninth example: 4 p and 2 s event (within 2026-06-03 07:03:00 and 2026-06-03 07:04:00)
start_filter_9 = dt.datetime(2026, 6, 3, 7, 3, 0)
end_filter_9 = dt.datetime(2026, 6, 3, 7, 4, 0)
subset_df_9 = event_df3[(event_df3['time_value'] >= start_filter_9) & (event_df3['time_value'] <= end_filter_9)].copy()
subset_df_9[['time_value', 'event_type', 'quality_associatedPhaseCount', 'quality_usedPhaseCount', 'quality_usedStationCount', 'quality_associatedStationCount']]

,time_value,event_type,quality_associatedPhaseCount,quality_usedPhaseCount,quality_usedStationCount,quality_associatedStationCount


In [23]:
# Tenth example: 3 p and 3 s event (within 2026-06-03 15:22:00 and 2026-06-03 15:23:00)
start_filter_10 = dt.datetime(2026, 6, 3, 15, 22, 0)
end_filter_10 = dt.datetime(2026, 6, 3, 15, 23, 0)
subset_df_10 = event_df3[(event_df3['time_value'] >= start_filter_10) & (event_df3['time_value'] <= end_filter_10)].copy()
subset_df_10[['time_value', 'event_type', 'quality_associatedPhaseCount', 'quality_usedPhaseCount', 'quality_usedStationCount', 'quality_associatedStationCount']]

,time_value,event_type,quality_associatedPhaseCount,quality_usedPhaseCount,quality_usedStationCount,quality_associatedStationCount


In [24]:
# Eleventh example: 10 used phases to locate, 11 total phases, 1 s phase discarded. Total 7 p and 4 s (within 2026-06-01 23:52:00 and 2026-06-01 23:53:00)
start_filter_11 = dt.datetime(2026, 6, 1, 23, 52, 0)
end_filter_11 = dt.datetime(2026, 6, 1, 23, 53, 0)
subset_df_11 = event_df3[(event_df3['time_value'] >= start_filter_11) & (event_df3['time_value'] <= end_filter_11)].copy()
subset_df_11[['time_value', 'event_type', 'quality_associatedPhaseCount', 'quality_usedPhaseCount', 'quality_usedStationCount', 'quality_associatedStationCount']]

,time_value,event_type,quality_associatedPhaseCount,quality_usedPhaseCount,quality_usedStationCount,quality_associatedStationCount


Summarizing the results for seiscomp6, we have:

| Event | p phases | s phases | quality_associatedPhaseCount | quality_usedPhaseCount | quality_usedStationCount | quality_associatedStationCount |
|------|----------|----------|------------------------------|------------------------|--------------------------|--------------------------------|
| 1    | 72       | 72       | 144                          | 130                    | 71                       | NaN                            |
| 2    | 4        | 4        | 8                            | 8                      | 4                        | NaN                            |
| 3    | 4        | 3        | 7                            | 7                      | 4                        | NaN                            |
| 4    | 4        | 2        | 6                            | 6                      | 4                        | NaN                            |
| 5    | 3        | 3        | 6                            | 6                      | 3                        | NaN                            |
| 6    | 7        | 4        | 11                           | 10                     | 7                        | NaN                            |

As we can see, the column _quality_usedStationCount_ specifies the total processed stations, a very useful quantity if I require to check that an event have at least 4 p processed. On the other hand, the column _quality_associatedStationCount_ is empty at least for the tests, therefore if it is not used on other quality checks later, it can de discarded.

In conclusion, at seiscomp6 it is possible to make a composed rule to look for locatable events. In particular if an event with _'not locatable'_ label have:

$$ quality\_usedStationCount \geq 4 \quad \text{and} \quad quality\_associatedPhaseCount \geq  quality\_usedStationCount + 2$$

then it can be flagged as locatable, and therefore it should be reviewed to check if the label is correct or not. For seiscomp3 we can use the previous verification based on:

$$ quality\_associatedPhaseCount \geq 8$$

and leave as a task if an improved version using the Arrive table can be implemented in the future, if the quality columns are not populated correctly.

Finally, the behavior of _quality_associatedStationCount_ and _quality_associatedPhaseCount_ at seiscomp6 allows to improve the criteria for those events that are not locatable but need one p or one s phase to be locatable: If an event have 3 p and at least 2 s phases, it requires only one p to be locatable, or if an event have 4 p and at least 1 s phase, it requires only one s to be locatable. Therefore, the criteria for locatable events can be further improved by checking the number of p and s phases associated to each event, and not only the total number of phases. This can be implemented by adding two rules for each case:

$$ quality\_usedStationCount == 3 \quad \text{and} \quad quality\_associatedPhaseCount \geq  quality\_usedStationCount + 2$$
$$ or $$
$$ quality\_usedStationCount \geq 4 \quad \text{and} \quad quality\_associatedPhaseCount \geq  quality\_usedStationCount + 1$$

Using the same structure, for seiscomp3 we can use the previous criteria:

$$ quality\_associatedPhaseCount \leq 7$$


# 2. Column to column comparisons

The not locatable check uses a composed rule based on the columns _quality_usedStationCount_ and _quality_associatedPhaseCount_ to identify events that are labeled as "not locatable" but have enough associated phases and processed stations to be potentially locatable. This check types stablish a new challenge, as it requires to make comparisons between two columns, and not only between a column and a threshold value. However, it is still possible to vectorize this check by applying the composed rule to the entire dataset at once, rather than iterating through each earthquake individually.

The idea here is to create a new general function that handles column to column comparisons, receiving the seismic data, the two columns to be compared, the mode of comparison, and any necessary offset if required by the rule. The function will then apply the comparison to the entire dataset and return a boolean mask indicating which earthquakes meet the criteria for being flagged.

In [25]:
def build_column_compare_mask(
    df: pd.DataFrame,
    left_col: str,
    op: str,
    right_col: str,
    offset: float = 0.0,
    factor: float = 1.0,
    dtype=np.float64,
) -> np.ndarray:
    left = df[left_col].to_numpy(dtype=dtype, copy=False)
    right = df[right_col].to_numpy(dtype=dtype, copy=False) * factor + offset

    ops = {
        "gt": np.greater,
        "ge": np.greater_equal,
        "lt": np.less,
        "le": np.less_equal,
        "eq": np.equal,
        "ne": np.not_equal,
    }
    return ops[op](left, right)

In [26]:
# Test the function for seiscomp6 events with the not locatable check
seiscomp6_events = event_df3[event_df3['time_value'] >= dt.datetime(2026, 3, 17)].copy()
time1 = time.time()
not_locatable_mask = build_column_compare_mask(
    df=seiscomp6_events,
    left_col='quality_associatedPhaseCount',
    op='ge',
    right_col='quality_usedStationCount',
    offset=2.0
)
potentially_locatable_events = seiscomp6_events[not_locatable_mask]
time2 = time.time()
print(f"Number of potentially locatable events: {len(potentially_locatable_events)}")
print(f"Time elapsed for column to column comparison check: {time2 - time1:.4f} seconds")

Number of potentially locatable events: 3472
Time elapsed for column to column comparison check: 0.0021 seconds


# 3. Non-numeric categorical checks

In the seismic revision routine, there are also some checks that are based on non-numeric categorical values, such as invalid event types or just non-labeled events. These checks not require to be vectorized, but we can return a boolean mask to maintain the same structure as the previous checks. For example, we can create a function that checks if the event type is in a list of invalid event types, and return a boolean mask indicating which events are flagged as invalid. This function can be applied to the entire dataset at once, and it can be easily integrated into the wrapper function for seismic quality checks.

In [27]:
def build_category_mask(
    df: pd.DataFrame,
    column: str,
    mode: str,
    values: list[str] | None = None,
) -> np.ndarray:
    s = df[column]

    if mode == "is_null":
        return s.isna().to_numpy()
    elif mode == "not_null":
        return s.notna().to_numpy()
    elif mode == "in":
        return s.isin(values).to_numpy()
    elif mode == "not_in":
        return (~s.isin(values)).to_numpy()
    else:
        raise ValueError(f"Unsupported category mode: {mode}")

In [28]:
# Check for events without event type
time1 = time.time()
no_event_type_mask = build_category_mask(
    df=event_df3,
    column='event_type',
    mode='is_null'
)
events_without_type = event_df3[no_event_type_mask]
time2 = time.time()
print(f"Number of events without event type: {len(events_without_type)}")
print(f"Time elapsed for category check: {time2 - time1:.4f} seconds")

Number of events without event type: 2
Time elapsed for category check: 0.0019 seconds


In [29]:
# Check for events with invalid event types
valid_types = ["earthquake", "not locatable", "volcanic eruption", "explosion", "not existing", "outside of network interest"]
time1 = time.time()
invalid_types = build_category_mask(
    df=event_df3,
    column='event_type',
    mode='not_in',
    values=valid_types)
events_with_invalid_type = event_df3[invalid_types]
time2 = time.time()
print(f"Number of events with invalid event type: {len(events_with_invalid_type)}")
print(f"Time elapsed for invalid event type check: {time2 - time1:.4f} seconds")

Number of events with invalid event type: 2
Time elapsed for invalid event type check: 0.0021 seconds


As you can see, using a simple not_in comparison in the entire dataframe, we can found both events with NaN event type and events with invalid event types in a very short time, which is much faster than iterating through each event individually and checking the event type one by one. This approach allows us to efficiently identify events that may have been misclassified or require further investigation based on their event type, while still maintaining readability and maintainability of the code.

By implementing these vectorized checks for numeric comparisons, column-to-column comparisons, and categorical checks, we can significantly improve the performance of the seismic quality checks while maintaining readability and maintainability of the code. The use of boolean masks allows us to apply the checks to the entire dataset at once, which is much faster than iterating through each event individually. Additionally, by using a configuration file for the checks, we can easily add or modify checks without changing the code of the wrapper function, making it more flexible and adaptable to different requirements.

As another example, consider all the events that has a comment, independently of the content of the comment. In the seismic revision routine, there is a check to identify events that have a comment different than "DESTACADO", which is used to flag events that may require further investigation. This check can be implemented using the `build_category_mask` function by checking for non-null values in the 'comment' column, as well as checking for comments that are not equal to "DESTACADO". However, there are two additional characteristics to be considered:

1. At seiscomp3, the 'DESTACADO' comment looks like b'DESTACADO', which is a byte string, while at seiscomp6 it looks like 'DESTACADO', which is a regular string. Therefore, the check needs to be able to handle both cases.
2. Sometimes when an analyst needs to remove a comment for any reason, they can just put an empty comment, which is not a null value, but it is also a valid comment. Therefore, the check needs to consider empty comments as well.

In [30]:
# Check how many events have non-null comment
time1 = time.time()
comment_mask = build_category_mask(
    df=event_df3,
    column='comment',
    mode='is_null'
)
events_with_comment = event_df3[~comment_mask]
time2 = time.time()
print(f"Number of events with comment: {len(events_with_comment)}")
print(f"Time elapsed for comment check: {time2 - time1:.4f} seconds")

Number of events with comment: 54
Time elapsed for comment check: 0.0012 seconds


In [31]:
# Print unique comments to check if there are comments different than 'DESTACADO'
unique_comments = events_with_comment['comment'].unique()
print(f"Unique comments: {unique_comments}")

Unique comments: ['DESTACADO' 'DESTACADO\n']


In [32]:
# Check for comments different than 'DESTACADO' (considering both byte string and regular string)
time1 = time.time()
destacado_mask = build_category_mask(
    df=events_with_comment,
    column='comment',
    mode='not_in',
    values=['DESTACADO', '', b'DESTACADO', b'']
)
events_with_non_destacado_comment = events_with_comment[destacado_mask]
time2 = time.time()
print(f"Number of events with non-DESTACADO comment: {len(events_with_non_destacado_comment)}")
print(f"Time elapsed for non-DESTACADO comment check: {time2 - time1:.4f} seconds")

Number of events with non-DESTACADO comment: 1
Time elapsed for non-DESTACADO comment check: 0.0043 seconds


# 4. Polygonal checks

Probably one of the most important checks in the seismic revision routine is related to be inside or outside different predefined polygons, which can be used to identify events that are outside of the network interest, or events that are located in specific areas of interest. These checks require to perform a point-in-polygon test for each event, which can be computationally expensive if done in a loop. However, it is possible to vectorize this check by using spatial indexing and efficient point-in-polygon algorithms, such as those provided by libraries like Shapely or GeoPandas. By creating a spatial index for the polygons and applying the point-in-polygon test to the entire dataset at once, we can significantly reduce the time taken to perform this check while still providing accurate results.

Let's create a function that performs a point-in-polygon test for a given set of polygons and a DataFrame of seismic events, and returns a boolean mask indicating which events are inside the polygons. This function can be integrated into the wrapper function for seismic quality checks to identify events that are outside of the network interest or located in specific areas of interest.

In [33]:
def build_polygon_mask(df, lon_col, lat_col, polygon, mode="inside"):
    pts = points(df[lon_col].to_numpy(), df[lat_col].to_numpy())
    inside = contains(polygon, pts)
    return inside if mode == "inside" else ~inside

Actually the polygons are stored inside the model_files folder at the main repository, and they can be loaded as Shapely polygons using the following code:

In [34]:
path_to_polygons = "../model_files/"
polygon_to_load = "Modelo_CARMA.txt"

# Load file: coordinates are comma-separated as lon,lat
coords = np.loadtxt(path_to_polygons + polygon_to_load, delimiter=",", skiprows=1)

# Build shapely polygon with (lon, lat)
polygon_created = Polygon(coords)

# Folium needs [lat, lon]
folium_coords = [[lat, lon] for lon, lat in coords]

# Center map on polygon centroid
centroid = polygon_created.centroid
m = folium.Map(
    location=[centroid.y, centroid.x],
    zoom_start=6,
    tiles="CartoDB positron"
)

# Add polygon
folium.Polygon(
    locations=folium_coords,
    color="red",
    weight=2,
    fill=True,
    fill_color="orange",
    fill_opacity=0.2,
    tooltip=polygon_to_load
).add_to(m)

# Optional centroid marker
folium.Marker(
    location=[centroid.y, centroid.x],
    popup=f"Centroid: ({centroid.y:.4f}, {centroid.x:.4f})"
).add_to(m)

m

Now, let's check how many events are inside this polygon using the `build_polygon_mask` function:

In [35]:
# Check how many events are inside the polygon
time1 = time.time()
polygon_mask = build_polygon_mask(
    df=event_df3,
    lon_col='longitude_value',
    lat_col='latitude_value',
    polygon=polygon_created,
    mode='inside'
)
events_inside_polygon = event_df3[polygon_mask]
time2 = time.time()
print(f"Number of events inside the polygon: {len(events_inside_polygon)}")
print(f"Time elapsed for polygonal check: {time2 - time1:.4f} seconds")

Number of events inside the polygon: 289
Time elapsed for polygonal check: 0.0276 seconds


However, as you can see, the time taken to perform the point-in-polygon test for all events is quite high, especially if the dataset is large. This is because the `contains` function from Shapely does not use any spatial indexing, and it needs to check each point against the polygon individually. To improve the performance of this check, we can use the `prepare` function from Shapely to create a prepared geometry for the polygon, which allows for faster point-in-polygon tests by building an internal spatial index on the edges of the polygon. By using a prepared geometry, we can significantly reduce the time taken to perform the point-in-polygon test while still providing accurate results:

In [36]:
def build_polygon_mask_2(df, lon_col, lat_col, polygon_i, mode="inside"):
    if not is_prepared(polygon_i):
        prepare(polygon_i)
    inside = shapely.contains_xy(
        polygon_i,
        df[lon_col].to_numpy(),
        df[lat_col].to_numpy()
    )
    return inside if mode == "inside" else ~inside

In [37]:
# Check speed of the second version of the polygon mask function
time1 = time.time()
polygon_mask_2 = build_polygon_mask_2(
    df=event_df3,
    lon_col='longitude_value',
    lat_col='latitude_value',
    polygon_i=polygon_created,
    mode='inside'
)
time2 = time.time()
print(f"Time elapsed for polygonal check with prepared polygon: {time2 - time1} seconds")
print(f"Number of events inside the polygon with prepared polygon: {polygon_mask_2.sum()}")

Time elapsed for polygonal check with prepared polygon: 0.0011551380157470703 seconds
Number of events inside the polygon with prepared polygon: 289


##  4.1. Example: Check if the event is inside/outside local zone and has wrong label

An example to apply this polygonal check is the case of events where the event type is "earthquake" or "volcanic eruption", but the event is located outside of the local zone defined by the polygon in the 'colom_ecu_fro.txt' file. This check can be implemented by creating a subset of the events that have event type "earthquake" or "volcanic eruption", and then applying the polygonal check to this subset to identify which events are outside of the local zone.

For this check we will consider two cases:
1. The event is outside the local zone (i.e., outside of the polygon defined in the 'colom_ecu_fro.txt' file) and event_type is NOT "outside of network interest" or "not locatable" or ''not existing''.
2. The event is inside the local zone (i.e., inside the polygon defined in the 'colom_ecu_fro.txt' file) and event_type is "outside of network interest".

In [38]:
# 1. Event is outside the local zone and event_type is "earthquake" or "volcanic eruption"
path_to_polygons = "../model_files/"
polygon_to_load = "colom_ecu_fro.txt"
coords = np.loadtxt(path_to_polygons + polygon_to_load, delimiter=",", skiprows=1)
local_zone_polygon = Polygon(coords)
time1 = time.time()
subset = event_df3[event_df3['event_type'].isin(["earthquake", "volcanic eruption"])]
polygon_mask = build_polygon_mask_2(
    df=subset,
    lon_col='longitude_value',
    lat_col='latitude_value',
    polygon_i=local_zone_polygon,
    mode='outside'
)
events_outside_local_zone = subset[polygon_mask]
time2 = time.time()
print(f"Number of events outside the local zone with event type 'earthquake' or 'volcanic eruption': {len(events_outside_local_zone)}")
print(f"Time elapsed for outside local zone check: {time2 - time1:.4f} seconds")

Number of events outside the local zone with event type 'earthquake' or 'volcanic eruption': 0
Time elapsed for outside local zone check: 0.0065 seconds


In [39]:
path_to_polygons = "../model_files/"
polygon_to_load = "colom_ecu_fro.txt"
coords = np.loadtxt(path_to_polygons + polygon_to_load, delimiter=",", skiprows=1)
local_zone_polygon = Polygon(coords)
time1 = time.time()
subset = event_df3[~event_df3['event_type'].isin(["outside of network interest", "not locatable", "not existing"])]
polygon_mask = build_polygon_mask_2(
    df=subset,
    lon_col='longitude_value',
    lat_col='latitude_value',
    polygon_i=local_zone_polygon,
    mode='outside'
)
events_outside_local_zone_2 = subset[polygon_mask]
time2 = time.time()
print(f"Number of events outside the local zone without valid labels: {len(events_outside_local_zone_2)}")
print(f"Time elapsed for outside local zone check: {time2 - time1:.4f} seconds")

Number of events outside the local zone without valid labels: 1
Time elapsed for outside local zone check: 0.0073 seconds


In [40]:
# 2. Event is inside the local zone and event_type is "outside of network interest"
time1 = time.time()
subset = event_df3[event_df3['event_type'] == "outside of network interest"]
polygon_mask = build_polygon_mask_2(
    df=subset,
    lon_col='longitude_value',
    lat_col='latitude_value',
    polygon_i=local_zone_polygon,
    mode='inside'
)
events_inside_local_zone = subset[polygon_mask]
time2 = time.time()
print(f"Number of events inside the local zone with event type 'outside of network interest': {len(events_inside_local_zone)}")
print(f"Time elapsed for inside local zone check: {time2 - time1:.4f} seconds")

Number of events inside the local zone with event type 'outside of network interest': 0
Time elapsed for inside local zone check: 0.0035 seconds


In [41]:
events_inside_local_zone

,time_value,publicID,depth_value,magnitude_value,quality_standardError,depth_uncertainty,latitude_uncertainty,longitude_uncertainty,quality_associatedPhaseCount,quality_usedPhaseCount,...,quality_associatedStationCount,event_type,creationInfo_agencyID,text,latitude_value,longitude_value,magnitude_type,methodID,earthModelID,comment


# 5. Temporal comparisons

Although temporal comparisons can be made in an easy way using pandas directly, it can also be implemented using a vectorized approach by creating a boolean mask based on the time column and the specified time range. The main advantage of defining this function is that allows to implement temporal comparisons directly from the TOML config file, which can be useful for checks that require to be applied only to specific time ranges, such as checking for events that occurred during a specific seismic sequence or checking for events that occurred during a specific time period of interest. By creating a function that handles temporal comparisons, we can easily integrate this type of check into the wrapper function for seismic quality checks, and apply it to the entire dataset at once.

In [42]:
def build_datetime_mask(
    df: pd.DataFrame,
    column: str,
    op: str,
    value: str,
) -> np.ndarray:
    left = pd.to_datetime(df[column], utc=False)
    right = pd.Timestamp(value)

    ops = {
        "gt": np.greater,
        "ge": np.greater_equal,
        "lt": np.less,
        "le": np.less_equal,
        "eq": np.equal,
        "ne": np.not_equal,
    }
    return ops[op](left.to_numpy(), right.to_datetime64())

In [43]:
# Check for events that occurred after 2026-05-01
time1 = time.time()
after_may_mask = build_datetime_mask(
    df=event_df3,
    column='time_value',
    op='ge',
    value='2026-05-01'
)
events_after_may = event_df3[after_may_mask]
time2 = time.time()
print(f"Number of events that occurred after 2026-05-01: {len(events_after_may)}")
print(f"Time elapsed for temporal comparison check: {time2 - time1:.4f} seconds")

Number of events that occurred after 2026-05-01: 5741
Time elapsed for temporal comparison check: 0.0083 seconds


# 6. Composed rules

Most of the checks in the revision routine consist on a composition of one or more conditions, which can be based on numeric comparisons, column-to-column comparisons, categorical checks, or polygonal checks. For example, the not locatable check is a composition of a column-to-column comparison and a numeric comparison. To implement these composed rules in a vectorized way, we can simply combine the boolean masks generated by each individual check using logical operators (e.g., & for AND, | for OR). This allows us to apply the composed rules to the entire dataset at once, without the need for loops or iterative checks.

Let's create a simple function to handle the composition of multiple boolean masks based on a specified logical operator, and then use this function to implement some examples:

In [44]:
def combine_masks(masks: list[np.ndarray], logic: str = "and") -> np.ndarray:
    if not masks:
        raise ValueError("No masks provided")
    if logic == "and":
        return np.logical_and.reduce(masks)
    elif logic == "or":
        return np.logical_or.reduce(masks)
    elif logic == "xor":
        if len(masks) != 2:
            raise ValueError("XOR logic requires exactly 2 masks")
        return np.logical_xor(masks[0], masks[1])
    else:
        raise ValueError(f"Unsupported logic: {logic!r}")

## 6.1. Example: Check if the event has not been processed by the user

To check if an event has not been processed by the user, we need to satisfy two conditions:

1. The 'creationInfo_author' should be inside ["scanloc", "scautoloc_reg", "scanlocbay", "AI_picker"]
2. The event['type'] must be non "not existing"
3. The 'creationInfo_agencyID' must be "SGC"

all joined with the 'and' condition. We can implement this check by creating three boolean masks for each condition, and then combining them using the `combine_masks` function:

In [45]:
# Check if the event has not been processed by the user
time1 = time.time()
# 1. creationInfo_author in ["scanloc", "scautoloc_reg", "scanlocbay", "AI_picker"]
author_mask = build_category_mask(
    df=event_df3,
    column='creationInfo_author',
    mode='in',
    values=["scanloc", "scautoloc_reg", "scanlocbay", "AI_picker"]
)
# 2. event_type not "not existing"
event_type_mask = build_category_mask(
    df=event_df3,
    column='event_type',
    mode='not_in',
    values=["not existing"]
)
# 3. creationInfo_agencyID is "SGC"
agency_mask = build_category_mask(
    df=event_df3,
    column='creationInfo_agencyID',
    mode='in',
    values=["SGC"]
)
# Combine masks with 'and' condition
not_processed_mask = combine_masks([author_mask, event_type_mask, agency_mask], logic="and")
events_not_processed = event_df3[not_processed_mask]
time2 = time.time()
print(f"Number of events that have not been processed by the user: {len(events_not_processed)}")
print(f"Time elapsed for not processed check: {time2 - time1:.4f} seconds")

Number of events that have not been processed by the user: 1
Time elapsed for not processed check: 0.0184 seconds


## 6.2. Example: Check if any event inside the Caribbean Plate and Pacific Plate zones has depth higher than 30 km

For this check, we need to satisfy three conditions:
1. The event is inside the Caribbean Plate or Pacific Plate zones, which are defined by the polygons in the 'zona_caribe_P.txt' and 'zona_Pacifico_P.txt' files.
2. The event has a depth higher than 30 km.
3. The event type is "earthquake" or "outside of network interest".

We can implement this check by creating a boolean mask for each condition, and then combining them using the `combine_masks` function:

In [46]:
# Load Caribbean Plate and Pacific Plate polygons
caribbean_polygon = Polygon(np.loadtxt(path_to_polygons + "zona_Caribe_P.txt", delimiter=",", skiprows=1))
pacific_polygon = Polygon(np.loadtxt(path_to_polygons + "zona_Pacifico_P.txt", delimiter=",", skiprows=1))
# Filter dataframe by event_type
subset = event_df3[event_df3['event_type'].isin(["earthquake", "outside of network interest"])]
# Check if any event inside the Caribbean Plate and Pacific Plate zones has depth higher than 30 km
time1 = time.time()
# 1. Event is inside the Caribbean Plate or Pacific Plate zones
caribbean_mask = build_polygon_mask_2(
    df=subset,
    lon_col='longitude_value',
    lat_col='latitude_value',
    polygon_i=caribbean_polygon,
    mode='inside'
)
pacific_mask = build_polygon_mask_2(
    df=subset,
    lon_col='longitude_value',
    lat_col='latitude_value',
    polygon_i=pacific_polygon,
    mode='inside'
)
zone_mask = combine_masks([caribbean_mask, pacific_mask], logic="or")
# 2. Event has depth higher than 30 km
depth_mask = build_quality_mask(
    events=subset,
    column='depth_value',
    mode='gt',
    threshold=30.0
)
# Combine masks with 'and' condition
final_mask = combine_masks([zone_mask, depth_mask], logic="and")
events_in_zone_with_depth = subset[final_mask]
time2 = time.time()
print(f"Number of events inside the Caribbean Plate and Pacific Plate zones with depth higher than 30 km: {len(events_in_zone_with_depth)}")
print(f"Time elapsed for combined check: {time2 - time1:.4f} seconds")

Number of events inside the Caribbean Plate and Pacific Plate zones with depth higher than 30 km: 0
Time elapsed for combined check: 0.0016 seconds


# 6.3. Example: Check if a 'DESTACADO' event has the correct label

At this section we need to check the following conditions:

1. The event is NOT, ["not locatable", "outside of network interest", "volcanic eruption", "explosion", "not existing"]
2. The event have magnitude higher than 4.0
3. The event have NOT a comment with the word "DESTACADO" (considering both byte string and regular string)

In [47]:
# Filter dataframe by event_type
subset = event_df3[~event_df3['event_type'].isin(["not locatable", "outside of network interest", "volcanic eruption", "explosion", "not existing"])]
# Check if a 'DESTACADO' event has the correct label
time1 = time.time()
# 1. Magnitude higher than 4.0
magnitude_mask = build_quality_mask(
    events=subset,
    column='magnitude_value',
    mode='gt',
    threshold=4.0
)
# 2. Comment with the word "DESTACADO" (considering both byte string and regular string)
destacado_mask = build_category_mask(
    df=subset,
    column='comment',
    mode='not_in',
    values=['DESTACADO', b'DESTACADO']
)
# Combine masks with 'and' condition
final_mask = combine_masks([magnitude_mask, destacado_mask], logic="and")
events_destacado_without_label = subset[final_mask]
time2 = time.time()
print(f"Number of 'DESTACADO' events without the correct label: {len(events_destacado_without_label)}")
print(f"Time elapsed for 'DESTACADO' check: {time2 - time1:.4f} seconds")

Number of 'DESTACADO' events without the correct label: 0
Time elapsed for 'DESTACADO' check: 0.0018 seconds


However, since 2026 those events with longitudes lower than -82.5 (adjusted for Colombian local area) have a higher limit for the magnitude, which is 5.0 instead of 4.0, due to the low capability of the local network to locate events in that area. Here we can use two different approachs:

1. Continue with the previous boolean mask for magnitude higher than 4.0, and add a mask for events with longitudes lower than -82.5 that have magnitude higher than 5.0, and then combine them with an 'or' condition.
2. Create a new boolean mask considering the longitude condition. Then join the 'DESTACADO' comment mask with the new magnitude mask, which considers the longitude condition, using an 'and' condition.

We will implement the second approach, as it is more efficient and easier to read:

$$ (longitude \leq -82.5 \quad \text{and} \quad magnitude > 5.0) \quad \text{or} \quad (longitude > -82.5 \quad \text{and} \quad magnitude > 4.0)$$

In [48]:
# Create a boolean mask considering the longitude condition for magnitude threshold
time1 = time.time()
longitude_mask = build_quality_mask(
    events=subset,
    column='longitude_value',
    mode='le',
    threshold=-82.5
)
magnitude_mask_5 = build_quality_mask(
    events=subset,
    column='magnitude_value',
    mode='gt',
    threshold=5.0
)
magnitude_mask_4 = build_quality_mask(
    events=subset,
    column='magnitude_value',
    mode='gt',
    threshold=4.0
)
# Combine longitude and magnitude masks with 'or' condition
magnitude_mask = combine_masks(
    masks=[combine_masks([longitude_mask, magnitude_mask_5], logic="and"),
           combine_masks([~longitude_mask, magnitude_mask_4], logic="and")],
    logic="or"
)
# Combine magnitude mask with 'DESTACADO' comment mask with 'and' condition
final_mask = combine_masks([magnitude_mask, destacado_mask], logic="and")
events_destacado_without_label = subset[final_mask]
time2 = time.time()
print(f"Number of 'DESTACADO' events without the correct label considering longitude condition: {len(events_destacado_without_label)}")
print(f"Time elapsed for 'DESTACADO' check with longitude condition: {time2 - time1:.4f} seconds")

Number of 'DESTACADO' events without the correct label considering longitude condition: 0
Time elapsed for 'DESTACADO' check with longitude condition: 0.0026 seconds


## 6.4. Example: Verify if 'DESTACADO' events inside NonLinLoc zone use the correct earth model

To check this condition, we need to verify the following:

1. The event has a comment with the word "DESTACADO" (considering both byte string and regular string)
2. The event is inside the NonLinLoc zone, which is defined by the polygon in the 'zona_nll.txt' file. However, it must not be inside the VMM zone, which is defined by the polygon in the 'zona_vmm.txt' file, as events inside the VMM zone should use a different earth model.
3. The event has the correct earth model specified by the 'earthModelID' column with value 'Poveda_et_al_2018'.
4. The creationInfo_agencyID should be "SGC", as the NonLinLoc locations are only performed for events processed by SGC.

In [49]:
# Load NonLinLoc and VMM polygons
nll_polygon = Polygon(np.loadtxt(path_to_polygons + "zona_nll.txt", delimiter=",", skiprows=1))
vmm_polygon = Polygon(np.loadtxt(path_to_polygons + "zona_vmm.txt", delimiter=",", skiprows=1))
# Check if 'DESTACADO' events inside NonLinLoc zone use the correct earth model
time1 = time.time()
# 1. Comment with the word "DESTACADO" (considering both byte string and regular string)
destacado_mask = build_category_mask(
    df=event_df3,
    column='comment',
    mode='in',
    values=['DESTACADO', b'DESTACADO']
)
# 2. Event is inside the NonLinLoc zone but not inside the VMM zone
nll_mask = build_polygon_mask_2(
    df=event_df3,
    lon_col='longitude_value',
    lat_col='latitude_value',
    polygon_i=nll_polygon,
    mode='inside'
)
vmm_mask = build_polygon_mask_2(
    df=event_df3,
    lon_col='longitude_value',
    lat_col='latitude_value',
    polygon_i=vmm_polygon,
    mode='outside'
)
zone_mask = combine_masks([nll_mask, vmm_mask], logic="and")
# 3. Earth model is 'Poveda_et_al_2018'
earth_model_mask = build_category_mask(
    df=event_df3,
    column='earthModelID',
    mode='not_in',
    values=['Poveda_et_al_2018']
)
# 4. Agency must be SGC
sgc_agency_mask = build_category_mask(
    df=event_df3,
    column='creationInfo_agencyID',
    mode='in',
    values=["SGC"]
)
# Combine masks with 'and' condition
destacado_without_nll = combine_masks([destacado_mask, zone_mask, earth_model_mask, sgc_agency_mask], logic="and")
time2 = time.time()
events_destacado_without_nll = event_df3[destacado_without_nll]
print(f"Number of 'DESTACADO' events inside NonLinLoc zone without the correct earth model: {len(events_destacado_without_nll)}")
print(f"Time elapsed for 'DESTACADO' events inside NonLinLoc zone check: {time2 - time1:.4f} seconds")

Number of 'DESTACADO' events inside NonLinLoc zone without the correct earth model: 1
Time elapsed for 'DESTACADO' events inside NonLinLoc zone check: 0.0068 seconds


## 6.5. Example: Check for events outside NLL zone with NLL earth model

This check is similar to the previous one, but in this case we need to check for events that are outside the NonLinLoc zone but have the 'Poveda_et_al_2018' earth model, which is only used for events inside the NonLinLoc zone. The conditions to check are:

1. The event is outside the NonLinLoc zone, which is defined by the polygon in the 'zona_nll.txt' file.
2. The event has the 'Poveda_et_al_2018' earth model specified

In [50]:
# 1. Event is outside the NonLinLoc zone
time1 = time.time()
nll_mask = build_polygon_mask_2(
    df=event_df3,
    lon_col='longitude_value',
    lat_col='latitude_value',
    polygon_i=nll_polygon,
    mode='outside'
)
# 2. Earth model is 'Poveda_et_al_2018'
earth_model_mask = build_category_mask(
    df=event_df3,
    column='earthModelID',
    mode='in',
    values=['Poveda_et_al_2018']
)
time2 = time.time()
# Combine masks with 'and' condition
outside_nll_with_nll_model = combine_masks([nll_mask, earth_model_mask], logic="and")
events_outside_nll_with_nll_model = event_df3[outside_nll_with_nll_model]
print(f"Number of events outside NonLinLoc zone with NonLinLoc earth model: {len(events_outside_nll_with_nll_model)}")
print(f"Time elapsed for events outside NonLinLoc zone with NonLinLoc earth model check: {time2 - time1:.4f} seconds")

Number of events outside NonLinLoc zone with NonLinLoc earth model: 3
Time elapsed for events outside NonLinLoc zone with NonLinLoc earth model check: 0.0028 seconds


## 6.6. Example: Check for locatable events

As we see in previous sections of this notebook, the locatable check is a composition of a column-to-column comparison and a numeric comparison. The conditions to check are:

In particular if an event with _'not locatable'_ label have:

$$ quality\_usedStationCount \geq 4 \quad \text{and} \quad quality\_associatedPhaseCount \geq  quality\_usedStationCount + 2$$

then it can be flagged as locatable, and therefore it should be reviewed to check if the label is correct or not. For seiscomp3 we can use the previous verification based on:

$$ quality\_associatedPhaseCount \geq 8$$

where the cutoff for sc3 and sc6 is 2026-03-17 00:00:00. We can implement this check by creating a boolean mask for the total condition, defined as:

$$ (event\_type = 'not locatable') \quad \text{and} \quad (quality\_usedStationCount \geq 4) \quad \text{and} \quad (quality\_associatedPhaseCount \geq  quality\_usedStationCount + 2) \quad \text{if} \quad time\_value \geq 2026-03-17 \quad 00:00:00$$

$$\text{or}$$

$$ (event\_type = 'not locatable') \quad \text{and} \quad (quality\_associatedPhaseCount \geq 8) \quad \text{if} \quad time\_value < 2026-03-17 \quad 00:00:00$$


*WARNING*: Since 11-03-2025 17:00 HL the not locatable check was modified for events inside volcanic zones (defined in the _bna_volcanic_files_ folder). Previously, all events inside these zones MUST have the not locatable label unless they have a comment with the word "DESTACADO". However, since that date, events inside these zones can be locatable and MUST have the SGC preferred origin.

In [51]:
# Check for locatable events
time1 = time.time()
# 1. Event type is 'not locatable'
not_locatable_mask = build_category_mask(
    df=event_df3,
    column='event_type',
    mode='in',
    values=['not locatable']
)
# 2. Time condition for seiscomp3 and seiscomp6
time_condition_mask = build_datetime_mask(
    df=event_df3,
    column='time_value',
    op='ge',
    value='2026-03-17 00:00:00'
)
# 3. Quality conditions for seiscomp6
used_station_mask = build_quality_mask(
    events=event_df3,
    column='quality_usedStationCount',
    mode='ge',
    threshold=4.0
)
associated_phase_mask = build_column_compare_mask(
    df=event_df3,
    left_col='quality_associatedPhaseCount',
    op='ge',
    right_col='quality_usedStationCount',
    offset=2.0
)
# 4. Quality condition for seiscomp3
associated_phase_mask_sc3 = build_quality_mask(
    events=event_df3,
    column='quality_associatedPhaseCount',
    mode='ge',
    threshold=8.0
)
# Combine masks for seiscomp6 and seiscomp3 conditions
sc6_condition = combine_masks([not_locatable_mask, time_condition_mask, used_station_mask, associated_phase_mask], logic="and")
sc3_condition = combine_masks([not_locatable_mask, ~time_condition_mask, associated_phase_mask_sc3], logic="and")
final_mask = combine_masks([sc6_condition, sc3_condition], logic="or")
locatable_events = event_df3[final_mask]
time2 = time.time()
print(f"Number of locatable events in seiscomp6: {len(event_df3[sc6_condition])}")
print(f"Number of locatable events in seiscomp3: {len(event_df3[sc3_condition])}")
print(f"Total number of locatable events: {len(locatable_events)}")
print(f"Time elapsed for locatable check: {time2 - time1:.4f} seconds")

Number of locatable events in seiscomp6: 1
Number of locatable events in seiscomp3: 0
Total number of locatable events: 1
Time elapsed for locatable check: 0.0107 seconds


## 6.7. Potentially locatable events

As explained earlier, the criteria for those events that are not locatable but need one p or one s phase to be locatable looks like:

If an event have 3 p and at least 2 s phases, it requires only one p to be locatable, or if an event have 4 p and at least 1 s phase, it requires only one s to be locatable:

$$ quality\_usedStationCount == 3 \quad \text{and} \quad quality\_associatedPhaseCount \geq  quality\_usedStationCount + 2$$
$$ or $$
$$ quality\_usedStationCount \geq 4 \quad \text{and} \quad quality\_associatedPhaseCount \geq  quality\_usedStationCount + 1$$

Using the same structure, for seiscomp3 we can use the previous stablished criteria:

$$ quality\_associatedPhaseCount \leq 7$$

In [52]:
# Check for potentially locatable events
time1 = time.time()
# 1. Event type is 'not locatable'
not_locatable_mask = build_category_mask(
    df=event_df3,
    column='event_type',
    mode='in',
    values=['not locatable']
)
# 2. Time condition for seiscomp3 and seiscomp6
time_condition_mask = build_datetime_mask(
    df=event_df3,
    column='time_value',
    op='ge',
    value='2026-03-17 00:00:00'
)
# 3. Quality conditions for seiscomp6
potentially_locatable_mask_1 = combine_masks([
    build_quality_mask(
        events=event_df3,
        column='quality_usedStationCount',
        mode='eq',
        threshold=3.0
    ),
    build_column_compare_mask(
        df=event_df3,
        left_col='quality_associatedPhaseCount',
        op='ge',
        right_col='quality_usedStationCount',
        offset=2.0
    )
], logic="and")
potentially_locatable_mask_2 = combine_masks([
    build_quality_mask(
        events=event_df3,
        column='quality_usedStationCount',
        mode='ge',
        threshold=4.0
    ),
    build_column_compare_mask(
        df=event_df3,
        left_col='quality_associatedPhaseCount',
        op='ge',
        right_col='quality_usedStationCount',
        offset=1.0
    )
], logic="and")
potentially_locatable_mask_sc6 = combine_masks([not_locatable_mask, time_condition_mask, combine_masks([potentially_locatable_mask_1, potentially_locatable_mask_2], logic="or")], logic="and")
# 4. Quality condition for seiscomp3
potentially_locatable_mask_sc3 = combine_masks([not_locatable_mask, ~time_condition_mask, build_quality_mask(
    events=event_df3,
    column='quality_associatedPhaseCount',
    mode='le',
    threshold=7.0
)], logic="and")
# Combine masks for seiscomp6 and seiscomp3 conditions
final_mask = combine_masks([potentially_locatable_mask_sc6, potentially_locatable_mask_sc3], logic="or")
potentially_locatable_events = event_df3[final_mask]
time2 = time.time()
print(f"Number of potentially locatable events in seiscomp6: {len(event_df3[potentially_locatable_mask_sc6])}")
print(f"Number of potentially locatable events in seiscomp3: {len(event_df3[potentially_locatable_mask_sc3])}")
print(f"Total number of potentially locatable events: {len(potentially_locatable_events)}")
print(f"Time elapsed for potentially locatable check: {time2 - time1:.4f} seconds")

Number of potentially locatable events in seiscomp6: 377
Number of potentially locatable events in seiscomp3: 0
Total number of potentially locatable events: 377
Time elapsed for potentially locatable check: 0.0119 seconds


## 6.8. Example: Check correspondence between zones and magnitude labels

At the RSNC we consider two types of polygons:

1. Seven for selecting magnitude_type: called zona[1-5].txt, zona_vmm.txt and zona_PtoGaitan, where magnitude_type should be 'MLr_[1-5]', 'MLr_vmm' and 'MLr_PtoGtn' respectively.
2. Four for selecting special earth models: zona_vmm.txt should have only events with 'VMM' earth model, Modelo_CARMA.txt with 'CARMA' earth model, zona_PtoGaitan.txt with 'Pto_Gaitan' earth model and Modelo_Cesar.txt with 'modelCesar2'.

The two polygon types are NOT equal, excepting for VMM and Pto_Gaitan zones: For example, it can exist events with CARMA model with MLr_3 or MLr_4 magnitudes, but there aren't events with VMM model with magnitudes different from MLr_vmm, or events with Pto_Gaitan model with magnitudes different from MLr_PtoGtn. Also, zona_vmm.txt is inside zona3.txt and zona_PtoGaitan.txt is inside zona5.txt, but the rest of the polygons are not nested.

Furthermore, the check must be applied in opposite cases: Check for any event outside the corresponding polygon with the corresponding magnitude type, and check for any event inside the corresponding polygon with a magnitude type different from the corresponding one. For example, for the zona3.txt polygon, we need to check for events outside this polygon with magnitude type 'MLr_3' (a mistake), and events inside this polygon with magnitude type different from 'MLr_3' (another mistake). This means that for each polygon we need to check for two different conditions, which can be implemented by creating two boolean masks for each polygon and magnitude type, and then combining them with an 'or' condition.

One last thing: This check must be applied for every event different from not existing and not locatable and if the event have a 'DESTACADO' comment, it can have a moment magnitud (Mw) fixed instead of the corresponding MLr magnitude, so those events with 'DESTACADO' comment should be considered from this check (MUST NOT be ignored due that they can have a fixed Mw instead of the corresponding MLr magnitude, but they also can have a MLr magnitude different from the corresponding one, which is also a mistake).

### 6.8.1. Zone3

The condition follows:

$$ ((magnitude\_type = MLr_3 \quad \text{and} \quad event \quad \text{is outside} \quad zona3) \quad \text{or} \quad (magnitude\_type \neq MLr_3 \quad \text{and} \quad event \quad \text{is inside} \quad zona3)) \quad \text{and} \quad (event\_type \neq 'not existing' \quad \text{and} \quad event\_type \neq 'not locatable')$$

and should be applied for events with event_type different from 'not existing' and 'not locatable'. Moreover, we must consider the 'DESTACADO' condition also, where events with this comment can have a moment magnitude (Mw) instead of the corresponding MLr magnitude, even if they are inside the corresponding polygon. This means that for events with 'DESTACADO' comment, the condition should be:

$$ If \quad comment \quad contains \quad 'DESTACADO' \quad then \quad magnitude\_type \quad in \quad 'Mw'$$

Summaryzing:

$$\text{Valid\_Event\_Type} \quad \mathbf{AND} \quad (\text{mlr3\_mask} \quad \mathbf{XOR} \quad \text{inside\_zona3\_mask}) \quad \mathbf{AND} \quad \mathbf{NOT} \quad (D \quad \mathbf{AND} \quad M)$$

where D is the condition that the comment contains 'DESTACADO' and M is the condition that the magnitude type is in ['MLr_3', 'Mw'].

In [53]:
time1 = time.time()
# First condition: Filter dataframe by event_type
subset = event_df3[~event_df3['event_type'].isin(['not existing', 'not locatable'])]
# Load zona3 polygon
zona3_polygon = Polygon(np.loadtxt(path_to_polygons + "zona3.txt", delimiter=",", skiprows=1))
# Second condition: inside_zone3_mask
is_inside_zone3 = build_polygon_mask_2(
        df=subset,
        lon_col='longitude_value',
        lat_col='latitude_value',
        polygon_i=zona3_polygon,
        mode='inside'
    )
# Third condition: mlr3_mask
have_mlr3 = build_category_mask(
        df=subset,
        column='magnitude_type',
        mode='in',
        values=['MLr_3']
    )
# Fourth condition: DESTACADO comment mask
destacado_mask = build_category_mask(
        df=subset,
        column='comment',
        mode='in',
        values=['DESTACADO', b'DESTACADO']
)
# Fifth condition: magnitude type in 'Mw'
mw_mask = build_category_mask(
        df=subset,
        column='magnitude_type',
        mode='in',
        values=['Mw']
)
# Combine conditions
nested_1 = combine_masks([have_mlr3, is_inside_zone3], logic="xor")  # mlr3_mask XOR is_inside_zone3
nested_2 = combine_masks([destacado_mask, mw_mask], logic="and")  # DESTACADO AND Mw
final_mask = combine_masks([nested_1, ~nested_2], logic="and")
events_with_magnitude_zone3_issue = subset[final_mask]
time2 = time.time()
print(f"Number of events with magnitude type and zona3 mismatch: {len(events_with_magnitude_zone3_issue)}")
print(f"Time elapsed for zona3 magnitude check: {time2 - time1:.4f} seconds")

Number of events with magnitude type and zona3 mismatch: 1
Time elapsed for zona3 magnitude check: 0.0386 seconds


### 6.8.2. Zone4

A similar approach can be followed for the zona4.txt polygon, where the magnitude type should be 'MLr_4'. Let be:

- I = Events inside zona4 polygon
- M = magnitude_type == 'MLr_4'
- V = valid event type, meaning event_type is neither 'not existing' nor 'not locatable'
- D = comment contains 'DESTACADO'
- A = magnitude_type == 'Mw'

Then the condition for zona4 is:

$$ V \quad \mathbf{AND} \quad (M \quad \mathbf{XOR} \quad I) \quad \mathbf{AND} \quad \mathbf{NOT} \quad (D \quad \mathbf{AND} \quad A)$$

where the first part of the condition is the same as for zona3, but changing the magnitude type to 'MLr_4', and the second part of the condition is also the same, where events with 'DESTACADO' comment can have a moment magnitude (Mw) instead of the corresponding MLr magnitude, even if they are inside the corresponding polygon.



In [54]:
time1 = time.time()
# First condition: Filter dataframe by event_type
subset = event_df3[~event_df3['event_type'].isin(['not existing', 'not locatable'])]
# Load zona3 polygon
zona4_polygon = Polygon(np.loadtxt(path_to_polygons + "zona4.txt", delimiter=",", skiprows=1))
# Second condition: inside_zone4_mask
is_inside_zone4 = build_polygon_mask_2(
        df=subset,
        lon_col='longitude_value',
        lat_col='latitude_value',
        polygon_i=zona4_polygon,
        mode='inside'
    )
# Third condition: mlr4_mask
have_mlr4 = build_category_mask(
        df=subset,
        column='magnitude_type',
        mode='in',
        values=['MLr_4']
    )
# Combine conditions
nested_1 = combine_masks([have_mlr4, is_inside_zone4], logic="xor")  # mlr4_mask XOR is_inside_zone4
nested_2 = combine_masks([destacado_mask, mw_mask], logic="and")  # DESTACADO AND Mw
final_mask = combine_masks([nested_1, ~nested_2], logic="and")
events_with_magnitude_zone4_issue = subset[final_mask]
time2 = time.time()
print(f"Number of events with magnitude type and zona4 mismatch: {len(events_with_magnitude_zone4_issue)}")
print(f"Time elapsed for zona4 magnitude check: {time2 - time1:.4f} seconds")

Number of events with magnitude type and zona4 mismatch: 2
Time elapsed for zona4 magnitude check: 0.0161 seconds


### 6.8.3. Zone2

As we see in previous sections, the format for checking regular zones follows the same format:

$$ V \quad \mathbf{AND} \quad (M \quad \mathbf{XOR} \quad I) \quad \mathbf{AND} \quad \mathbf{NOT} \quad (D \quad \mathbf{AND} \quad A)$$

where V is the valid event type condition, M is the magnitude type condition, I is the inside polygon condition, D is the 'DESTACADO' comment condition and A is the magnitude type in 'Mw' condition. However, for the zone 2 we have an extra condition: The VMM polygon is inside the zona2 polygon, but events inside the VMM polygon should have magnitude type 'MLr_vmm' instead of 'MLr_2', so we need to add another condition for events inside the VMM polygon with magnitude type 'MLr_vmm'. As we only want to check events for zone2 discarding events inside VMM zone, we can create a mask for events inside zona2 but outside VMM, and then combine it with the magnitude type condition with a 'xor' condition, and follows the same structure as before:

$$ I_{2,outer} \quad = \quad I_{2} \quad \mathbf{AND} \quad \mathbf{NOT} \quad I_{VMM}$$

and finally:

$$ V \quad \mathbf{AND} \quad (M \quad \mathbf{XOR} \quad I_{2,outer}) \quad \mathbf{AND} \quad \mathbf{NOT} \quad (D \quad \mathbf{AND} \quad A)$$

In [55]:
time1 = time.time()
# Load zona2 polygon
zona4_polygon = Polygon(np.loadtxt(path_to_polygons + "zona2.txt", delimiter=",", skiprows=1))
# Load zonavmm polygon
zonavmm_polygon = Polygon(np.loadtxt(path_to_polygons + "zona_vmm.txt", delimiter=",", skiprows=1))
# Second condition: inside_zone2_mask
is_inside_zone2 = build_polygon_mask_2(
        df=subset,
        lon_col='longitude_value',
        lat_col='latitude_value',
        polygon_i=zona4_polygon,
        mode='inside'
    )
# Third condition: inside_vmm_mask
is_inside_vmm = build_polygon_mask_2(
        df=subset,
        lon_col='longitude_value',
        lat_col='latitude_value',
        polygon_i=zonavmm_polygon,
        mode='inside'
    )
# Fourth condition: inside_zone2_outer_mask
is_inside_zone2_outer = combine_masks([is_inside_zone2, ~is_inside_vmm], logic="and")
# Fifth condition: mlr2_mask
have_mlr2 = build_category_mask(
        df=subset,
        column='magnitude_type',
        mode='in',
        values=['MLr_2']
    )
# Combine conditions
nested_1 = combine_masks([have_mlr2, is_inside_zone2_outer], logic="xor")  # mlr2_mask XOR is_inside_zone2_outer
nested_2 = combine_masks([destacado_mask, mw_mask], logic="and")  # DESTACADO AND Mw
final_mask = combine_masks([nested_1, ~nested_2], logic="and")
events_with_magnitude_zone2_issue = subset[final_mask]
time2 = time.time()
print(f"Number of events with magnitude type and zona2 mismatch: {len(events_with_magnitude_zone2_issue)}")
print(f"Time elapsed for zona2 magnitude check: {time2 - time1:.4f} seconds")

Number of events with magnitude type and zona2 mismatch: 4
Time elapsed for zona2 magnitude check: 0.0154 seconds


As you can see from the solutions, this check does NOT take account for events inside volcanic zones.

### 6.8.5. Zone5

Zone 5 follows the same format as zone 2, but in this case the inner polygon is zona_PtoGaitan.txt and the magnitude type for events inside this inner polygon should be 'MLr_PtoGtn' instead of 'MLr_5'. So:

In [56]:
time1 = time.time()
# Load zona5 polygon
zona5_polygon = Polygon(np.loadtxt(path_to_polygons + "zona5.txt", delimiter=",", skiprows=1))
# Load zona Pto Gaitan polygon
zonaPtoGtn_polygon = Polygon(np.loadtxt(path_to_polygons + "zona_PtoGaitan.txt", delimiter=",", skiprows=1))
is_inside_zone5 = build_polygon_mask_2(
        df=subset,
        lon_col='longitude_value',
        lat_col='latitude_value',
        polygon_i=zona5_polygon,
        mode='inside'
    )
is_inside_PtoGtn = build_polygon_mask_2(
        df=subset,
        lon_col='longitude_value',
        lat_col='latitude_value',
        polygon_i=zonaPtoGtn_polygon,
        mode='inside'
    )
is_inside_zone5_outer = combine_masks([is_inside_zone5, ~is_inside_PtoGtn], logic="and")
# Fifth condition: mlr2_mask
have_mlr5 = build_category_mask(
        df=subset,
        column='magnitude_type',
        mode='in',
        values=['MLr_5']
    )
# Combine conditions
nested_1 = combine_masks([have_mlr5, is_inside_zone5_outer], logic="xor")  # mlr2_mask XOR is_inside_zone2_outer
nested_2 = combine_masks([destacado_mask, mw_mask], logic="and")  # DESTACADO AND Mw
final_mask = combine_masks([nested_1, ~nested_2], logic="and")
events_with_magnitude_zone5_issue = subset[final_mask]
time2 = time.time()
print(f"Number of events with magnitude type and zona2 mismatch: {len(events_with_magnitude_zone5_issue)}")
print(f"Time elapsed for zona2 magnitude check: {time2 - time1:.4f} seconds")

Number of events with magnitude type and zona2 mismatch: 1
Time elapsed for zona2 magnitude check: 0.0639 seconds


### 6.8.6 Pto Gaitan zone

In [57]:
time1 = time.time()
# Third condition: mlr4_mask
have_mlrPtoGtn = build_category_mask(
        df=subset,
        column='magnitude_type',
        mode='in',
        values=['MLr_PtoGtn']
    )
# Combine conditions
nested_1 = combine_masks([have_mlrPtoGtn, is_inside_PtoGtn], logic="xor")  # mlr4_mask XOR is_inside_zone4
nested_2 = combine_masks([destacado_mask, mw_mask], logic="and")  # DESTACADO AND Mw
final_mask = combine_masks([nested_1, ~nested_2], logic="and")
events_with_magnitude_zonePtoGtn_issue = subset[final_mask]
time2 = time.time()
print(f"Number of events with magnitude type and zona PtoGtn mismatch: {len(events_with_magnitude_zonePtoGtn_issue)}")
print(f"Time elapsed for zonaPtoGtn magnitude check: {time2 - time1:.4f} seconds")

Number of events with magnitude type and zona PtoGtn mismatch: 0
Time elapsed for zonaPtoGtn magnitude check: 0.0027 seconds


# Additional checks

Through this notebook we have added some additional checks that are not included in the previous version of the seismic routine (i.e., the M>5 events with long < -82). However, there are some other checks that can be added to the routine to improve the quality of the review process. This section is intended to be a brainstorming of potential checks that can be added to the routine, but they are not implemented yet.

## 1. DESTACADO events with M<3 and outside interest network

It is almost impossible that an event with magnitude lower than 3 and outside network interest (defined in colom_ecu_fro.txt) will be a DESTACADO event, as these events are usually not significant events in terms of impact and felt reports, and therefore they are usually not felt and do not cause damage. Let's implement a check to identify these events, which can be a mistake in the labeling of the event or in the magnitude estimation.

In [58]:
time1 = time.time()
# First condition: DESTACADO comment mask (used from before)
# Second condition: Filter by magnitude
mag_less_than_3 = build_quality_mask(
    events=subset,
    column='magnitude_value',
    mode='lt',
    threshold=3.0
)
# Third condition: Filter by outside of network interest
outside_network_polygon = Polygon(np.loadtxt(path_to_polygons + "colom_ecu_fro.txt", delimiter=",", skiprows=1))
outside_network_mask = build_polygon_mask_2(
    df=subset,
    lon_col='longitude_value',
    lat_col='latitude_value',
    polygon_i=outside_network_polygon,
    mode='outside'
)
# Combine conditions
final_mask = combine_masks([destacado_mask, mag_less_than_3, outside_network_mask], logic="and")
destacado_events_with_mag_less_than_3_outside_network = subset[final_mask]
time2 = time.time()
print(f"Number of DESTACADO events with magnitude less than 3 and outside network interest: {len(destacado_events_with_mag_less_than_3_outside_network)}")
print(f"Time elapsed for DESTACADO events with M<3 and outside network check: {time2 - time1:.4f} seconds")

Number of DESTACADO events with magnitude less than 3 and outside network interest: 0
Time elapsed for DESTACADO events with M<3 and outside network check: 0.0061 seconds


## Example: Angel request -->> Check how many earthquakes and volcanic eruptions are inside volcanic zones (all saved in bna_volcanic_files folder)

As another example of the vectorization power let's consider the following case: We want to check how many earthquakes and volcanic eruptions are inside the volcanic zones defined in the bna_volcanic_files folder. This is a simple check that can be done by creating a boolean mask for each polygon and then combining them with an 'or' condition. However, we need to consider that the number of polygons can be large, so we need to use a loop to iterate over the files in the folder and create the masks.

The bna_volcanic_files folder contains 9 different files (labeled obspas{i}.txt) for Pasto, Nariño. Also one single file for Manizales (obsman.txt) and two main (obspop.txt and obspopvnh.txt) for Popayán, Cauca. The idea is to check two conditions for all events:

- If the event type is 'earthquake' or 'volcanic eruption'
- If the event is inside any of the volcanic zones defined in the bna_volcanic_files folder. This can be done by creating a boolean mask for each polygon and then combining them with an 'or' condition. However, we need to consider that the number of polygons can be large, so we need to use a loop to iterate over the files in the folder and create the masks

In [59]:
# Filter events for year 2026
events_2026 = event_df3[event_df3['time_value'].dt.year == 2026]
# Filter events for 'earthquake' or 'volcanic eruption'
events_eq_ve = events_2026[events_2026['event_type'].isin(['earthquake', 'volcanic eruption'])]
# Create a boolean mask for events inside volcanic zones
volcanic_mask = np.zeros(len(events_eq_ve), dtype=bool)
# Iterate over the files in the bna_volcanic_files folder
path_to_polygons = "../"
for file in os.listdir(path_to_polygons + "bna_volcanic_files/"):
    if file.endswith(".txt"):
        # Load the polygon from the file
        polygon = Polygon(np.loadtxt(path_to_polygons + "bna_volcanic_files/" + file, delimiter=",", skiprows=1))
        # Create a boolean mask for events inside the polygon
        inside_polygon_mask = build_polygon_mask_2(
            df=events_eq_ve,
            lon_col='longitude_value',
            lat_col='latitude_value',
            polygon_i=polygon,
            mode='inside'
        )
        # Combine the mask with the volcanic_mask using 'or' condition
        volcanic_mask = combine_masks([volcanic_mask, inside_polygon_mask], logic="or")
# Filter events for those inside volcanic zones
events_inside_volcanic_zones = events_eq_ve[volcanic_mask]

In [60]:
# Print number of events inside volcanic zones
print(f"Number of earthquakes and volcanic eruptions inside volcanic zones in 2026: {len(events_inside_volcanic_zones)}")

Number of earthquakes and volcanic eruptions inside volcanic zones in 2026: 21


In [61]:
events_inside_volcanic_zones

,time_value,publicID,depth_value,magnitude_value,quality_standardError,depth_uncertainty,latitude_uncertainty,longitude_uncertainty,quality_associatedPhaseCount,quality_usedPhaseCount,...,quality_associatedStationCount,event_type,creationInfo_agencyID,text,latitude_value,longitude_value,magnitude_type,methodID,earthModelID,comment
71,2026-08-10 05:32:10,SGC2026pqcmtq,9.843471,0.939660,0.495416,2.665479,0.892089,1.210738,28.0,28.0,...,NaN,earthquake,SGC,"Santa Rosa de Cabal - Risaralda, Colombia",4.843939,-75.504126,MLr_2,NonLinLoc,Poveda_et_al_2018,None
156,2026-08-10 12:42:30,SGC2026pqqtoz,0.010714,1.287484,0.505206,NaN,4.327261,4.416610,8.0,8.0,...,NaN,earthquake,SGC,"Villahermosa - Tolima, Colombia",4.905804,-75.241503,MLr_2,NonLinLoc,Poveda_et_al_2018,None
283,2026-08-10 17:01:24,SGC2026pqzirp,2.020000,1.219430,0.320000,0.400000,0.636396,0.636396,20.0,20.0,...,NaN,earthquake,SGC,"Villahermosa - Tolima, Colombia",4.894833,-75.303500,MLr_2,Hypo71,RSNC,None
591,2026-08-11 09:18:18,SGC2026psfsms,0.000000,1.398426,0.120000,1.000000,0.636396,0.636396,12.0,12.0,...,NaN,earthquake,SGC,"Pasto - Nariño, Colombia",1.193167,-77.234833,MLr_2,Hypo71,RSNC,None
1181,2026-08-13 05:03:35,SGC2026pvostv,11.576172,1.179581,0.455926,3.064358,1.372566,2.828864,14.0,14.0,...,NaN,earthquake,SGC,"Sotará (Paispamba) - Cauca, Colombia",2.261462,-76.579530,MLr_2,NonLinLoc,Poveda_et_al_2018,None
1262,2026-08-13 08:59:54,SGC2026pvwokt,5.800502,1.371038,0.452372,3.392701,2.200679,1.932799,22.0,18.0,...,NaN,earthquake,SGC,"Pasto - Nariño, Colombia",1.267389,-77.287499,MLr_2,NonLinLoc,Poveda_et_al_2018,None
1396,2026-08-13 19:13:44,SGC2026pwqxje,16.650000,1.373597,0.300000,4.000000,4.030509,4.030509,8.0,8.0,...,NaN,earthquake,SGC,"Sotará (Paispamba) - Cauca, Colombia",2.165000,-76.571167,MLr_2,Hypo71,RSNC,None
1548,2026-08-14 06:25:58,SGC2026pxneqo,8.572824,1.728305,0.660145,3.989473,2.681657,1.706902,24.0,20.0,...,NaN,earthquake,SGC,"Buesaco - Nariño, Colombia",1.268446,-77.208861,MLr_2,NonLinLoc,Poveda_et_al_2018,None
2286,2026-08-16 17:41:06,SGC2026qcbaej,4.090000,0.904819,0.270000,2.000000,0.707107,0.707107,12.0,12.0,...,NaN,earthquake,SGC,"Ibagué - Tolima, Colombia",4.517667,-75.399500,MLr_2,Hypo71,RSNC,None
2377,2026-08-17 01:20:08,SGC2026qcqfsr,2.420000,0.576436,0.310000,1.100000,1.909188,1.909188,10.0,10.0,...,NaN,earthquake,SGC,"Murillo - Tolima, Colombia",4.886500,-75.290667,MLr_2,Hypo71,RSNC,None


## Example: Freddy request --> Check how many events have been detected inside the Bucaramanga nest in 2026

The request here is simple: Consider events that are earthquakes, have been occurred since 2024 and are between two lat-lon limits (called lat_min, lat_max, lon_min, lon_max). Moreover, it must have depths inside 120 km and 180 km. The idea is to create a boolean mask for each condition and then combine them with an 'and' condition.

In [62]:
# Filter events since 2024
events_2024 = event_df3[event_df3['time_value'] >= '2021-08-26']
# Filter events for 'earthquake'
events_eq_ve = events_2024[events_2024['event_type'].isin(['earthquake'])]
print(f'Total earthquakes within 2024: {len(events_eq_ve)}')
# Define limits
lat_min, lat_max, lon_min, lon_max = 6.0, 8.0, -74.0, -72.0
# Create a boolean mask for events inside the Bucaramanga nest
lat_mask = build_quality_mask(
    events=events_eq_ve,
    column='latitude_value',
    mode='between',
    lower=lat_min,
    upper=lat_max
)
lon_mask = build_quality_mask(
    events=events_eq_ve,
    column='longitude_value',
    mode='between',
    lower=lon_min,
    upper=lon_max
)
depth_mask = build_quality_mask(
    events=events_eq_ve,
    column='depth_value',
    mode='between',
    lower=120.0,
    upper=180.0
)
combined_mask = combine_masks([lat_mask, lon_mask, depth_mask], logic="and")
bucaramanga_nest_events = events_eq_ve[combined_mask]
print(f'Total earthquakes inside Bucaramanga nest in 2024: {len(bucaramanga_nest_events)}')

Total earthquakes within 2024: 2544
Total earthquakes inside Bucaramanga nest in 2024: 593


## Example: Freddy request --> What the number of earthquakes per day in average over the last 5 years?

The request here is simple: Consider events that are earthquakes, are inside the colom_ecu_fro.txt polygon and have been occured since 2021-08-26. The idea is to create a boolean mask for each condition and then combine them with an 'and' condition. Then, we can group the events by day and count the number of events per day, and finally calculate the average number of events per day.

In [65]:
# Filter events since 2021-08-26
events_2021 = event_df3[event_df3['time_value'] >= '2021-08-26']
# Filter events for 'earthquake'
events_eq = events_2021[events_2021['event_type'].isin(['earthquake'])]
# Load colom_ecu_fro polygon
colom_ecu_fro_polygon = Polygon(np.loadtxt(path_to_polygons + "/model_files/colom_ecu_fro.txt", delimiter=",", skiprows=1))
# Create a boolean mask for events inside the colom_ecu_fro polygon
inside_colom_ecu_fro_mask = build_polygon_mask_2(
    df=events_eq,
    lon_col='longitude_value',
    lat_col='latitude_value',
    polygon_i=colom_ecu_fro_polygon,
    mode='inside'
)
# Filter events for those inside the colom_ecu_fro polygon
events_inside_colom_ecu_fro = events_eq[inside_colom_ecu_fro_mask]
print(f'Total earthquakes inside colom_ecu_fro polygon in the last 5 years: {len(events_inside_colom_ecu_fro)}')
# Group events by day and count the number of events per day
events_per_day = events_inside_colom_ecu_fro.groupby(events_inside_colom_ecu_fro['time_value'].dt.date).size()
# Calculate the average number of events per day
average_events_per_day = events_per_day.mean()
print(f'Average number of earthquakes per day in the last 5 years: {average_events_per_day:.2f}')

Total earthquakes inside colom_ecu_fro polygon in the last 5 years: 2544
Average number of earthquakes per day in the last 5 years: 141.33


# Example: Check how the average changes every year since 2021-08-26

The idea is to group earthquakes by year and then calculate the average number of them per day for each year. We can use the same boolean mask for events inside the colom_ecu_fro polygon and then group the events by year and count the number of events per day for each year.

In [66]:
def one_year_average(events_df, year):
    # Filter events for the given year
    events_year = events_df[events_df['time_value'].dt.year == year]
    print('There are {} events in {}'.format(len(events_year), year))
    # Filter events for 'earthquake'
    events_eq = events_year[events_year['event_type'].isin(['earthquake'])]
    # Create a boolean mask for events inside the colom_ecu_fro polygon
    inside_colom_ecu_fro_mask = build_polygon_mask_2(
        df=events_eq,
        lon_col='longitude_value',
        lat_col='latitude_value',
        polygon_i=colom_ecu_fro_polygon,
        mode='inside'
    )
    # Filter events for those inside the colom_ecu_fro polygon
    events_inside_colom_ecu_fro = events_eq[inside_colom_ecu_fro_mask]
    # Group events by day and count the number of events per day
    events_per_day = events_inside_colom_ecu_fro.groupby(events_inside_colom_ecu_fro['time_value'].dt.date).size()
    # Calculate the average number of events per day
    average_events_per_day = events_per_day.mean()
    return average_events_per_day

for year in range(2021, 2027):
    avg = one_year_average(event_df3, year)
    print(f'Average number of earthquakes per day in {year}: {avg:.2f}')

There are 0 events in 2021
Average number of earthquakes per day in 2021: nan
There are 0 events in 2022
Average number of earthquakes per day in 2022: nan
There are 0 events in 2023
Average number of earthquakes per day in 2023: nan
There are 0 events in 2024
Average number of earthquakes per day in 2024: nan
There are 0 events in 2025
Average number of earthquakes per day in 2025: nan
There are 5741 events in 2026
Average number of earthquakes per day in 2026: 141.33


# Requested example: Check the highest Chocó Events since 2026-08-10

It is required to search the 10 highest events in Chocó region since 2026-08-10. The rregion is defined between 3.5 to 5.5, -75 to -78 and from 0 to 70 km of depth. The idea is to create a boolean mask for each condition and then combine them with an 'and' condition. Then, we can sort the events by magnitude and select the top 10 events.

In [75]:
# Filter events since 2021-08-26
events_2021 = event_df3[event_df3['time_value'] >= '2026-08-09']
# Filter events for 'earthquake'
events_eq = events_2021[events_2021['event_type'].isin(['earthquake'])]
# Create a boolean mask for the Chocó region
lat_min, lat_max, lon_min, lon_max = 3.5, 5.5, -78.0, -75.0
# Create a boolean mask for events inside the Bucaramanga nest
lat_mask = build_quality_mask(
    events=events_eq_ve,
    column='latitude_value',
    mode='between',
    lower=lat_min,
    upper=lat_max
)
lon_mask = build_quality_mask(
    events=events_eq_ve,
    column='longitude_value',
    mode='between',
    lower=lon_min,
    upper=lon_max
)
depth_mask = build_quality_mask(
    events=events_eq_ve,
    column='depth_value',
    mode='between',
    lower=0.0,
    upper=70.0
)
magnitude_mask = build_quality_mask(
    events=events_eq_ve,
    column='magnitude_value',
    mode='ge',
    threshold=2.9)
combined_mask = combine_masks([lat_mask, lon_mask, depth_mask, magnitude_mask], logic="and")
Choco_events = events_eq_ve[combined_mask]
# Sort events by magnitude and select the top 10 events
top_10_Choco_events = Choco_events.sort_values(by='magnitude_value', ascending=False)
print(f'Top 10 highest events in Chocó region since 2026-08-10:')
top_10_Choco_events[['time_value', 'latitude_value', 'longitude_value', 'depth_value', 'magnitude_value', 'event_type']]

Top 10 highest events in Chocó region since 2026-08-10:


,time_value,latitude_value,longitude_value,depth_value,magnitude_value,event_type
5732,2026-08-27 15:25:26,4.536728,-76.741989,42.938058,4.422868,earthquake
4787,2026-08-24 18:55:25,4.407181,-76.751450,45.999163,4.120757,earthquake
5735,2026-08-27 16:21:19,4.570568,-76.778403,43.169085,4.015814,earthquake
4541,2026-08-24 02:40:57,4.445780,-76.730866,56.337612,3.786042,earthquake
4536,2026-08-24 02:28:05,4.455298,-76.717061,54.835938,3.764842,earthquake
...,...,...,...,...,...,...
4609,2026-08-24 06:15:36,4.638250,-76.723334,36.931362,2.941178,earthquake
4031,2026-08-22 07:08:11,4.464816,-76.794903,39.703683,2.931534,earthquake
3774,2026-08-21 08:28:07,4.407710,-76.722149,45.941406,2.926944,earthquake
689,2026-08-11 19:03:57,4.166594,-76.757194,53.334263,2.913022,earthquake
